In [1]:
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"

ORDERING_DATA_PATH = DATA_DIR / "ordering_processed"

In [2]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: e:\punctuation_project\.venv\Scripts\python.exe
PyTorch: 2.14.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
print("Project exists:", PROJECT_DIR.exists())
print("Ordering dataset exists:", ORDERING_DATA_PATH.exists())
print("Model checkpoint exists:",
      (MODEL_DIR / "sentence_ordering_transformer" / "model_state_dict.pt").exists())

Project exists: True
Ordering dataset exists: True
Model checkpoint exists: True


In [11]:
from pathlib import Path
from datasets import load_from_disk

PROJECT_DIR = Path(r"E:\punctuation_project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

ORDERING_PATH = PROCESSED_DIR / "ordering_processed"

print("Dataset path:", ORDERING_PATH)
print("Dataset exists:", ORDERING_PATH.exists())

ordering_data = load_from_disk(str(ORDERING_PATH))

print("\nDataset structure:")
print(ordering_data)

print("\nTraining features:")
print(ordering_data["train"].features)

print("\nOne training example:")
print(ordering_data["train"][0])

Dataset path: E:\punctuation_project\data\processed\ordering_processed
Dataset exists: True

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14845
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15431
    })
})

Training features:
{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}

One training example:
{'input_ids': [259, 112233, 5411, 267, 320, 353, 267, 19180, 996, 64981, 844, 274, 49046, 34376, 7218, 264, 111362, 165321, 259, 7479, 271, 2047, 31955, 52180, 260, 10561, 260, 7940, 260, 320, 338, 267, 259, 4620, 52180, 224293, 10561, 224293, 7940, 224293, 32901, 179147, 3547, 9400, 24031, 44248, 641, 9031, 12268, 844, 259, 23066, 830, 378, 320, 328, 267, 259, 4620, 6613, 1

In [13]:

from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer


# Project paths
PROJECT_DIR = Path(r"E:\punctuation_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"

ORDERING_DATA_PATH = DATA_DIR / "ordering_processed"

# Check GPU
print("=" * 60)
print("GPU INFORMATION")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
            2
        ),
        "GB"
    )

# Load existing dataset
print("\nLoading sentence-ordering dataset...")

ordering_data = load_from_disk(str(ORDERING_DATA_PATH))

print(ordering_data)

# Load the tokenizer used during preprocessing
TOKENIZER_NAME = "google/mt5-small"

ordering_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# Inspect three examples
for i in range(3):
    example = ordering_data["train"][i]

    input_text = ordering_tokenizer.decode(
        example["input_ids"],
        skip_special_tokens=False
    )

    target_text = ordering_tokenizer.decode(
        example["labels"],
        skip_special_tokens=False
    )

    print("\n" + "=" * 60)
    print(f"EXAMPLE {i + 1}")
    print("=" * 60)

    print("\nINPUT:")
    print(input_text)

    print("\nEXPECTED OUTPUT:")
    print(target_text)

    print("\nInput token count:", len(example["input_ids"]))
    print("Target token count:", len(example["labels"]))

print("\nTokenizer vocabulary size:", ordering_tokenizer.vocab_size)


GPU INFORMATION
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total GPU memory: 4.0 GB

Loading sentence-ordering dataset...
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14845
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15431
    })
})

EXAMPLE 1

INPUT:
restore order: S1: प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी. S2: यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है। S3: यह बिहार प्रान्त में स्थित प्रतिष्ठित अभियांत्रिकी संस्थानों में से एक है जिसमें विभिन्न शाखाएं है। S4: रोहिलखंड विश्वविद्यालय के स्वयं के अभियांत्रिकी प्रवेश परीक्षा के माध्यम से अखिल भारतीय स्तर पर आयोजित किया गया था। S5: इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है

In [12]:

import re
import numpy as np
from collections import Counter

# Check target format
def extract_order(target_ids):
    text = ordering_tokenizer.decode(
        target_ids,
        skip_special_tokens=True
    )

    sentence_ids = re.findall(r"S([1-5])", text)

    if len(sentence_ids) != 5:
        return None

    order = [int(x) - 1 for x in sentence_ids]

    if sorted(order) != [0, 1, 2, 3, 4]:
        return None

    return order


# Validate all splits
for split in ["train", "validation", "test"]:

    valid = 0
    invalid = 0
    lengths = []

    for example in ordering_data[split]:

        order = extract_order(example["labels"])

        if order is None:
            invalid += 1
        else:
            valid += 1

        lengths.append(len(example["input_ids"]))

    print(f"\n{split.upper()}")
    print("Valid examples:", valid)
    print("Invalid examples:", invalid)
    print("Maximum input length:", max(lengths))
    print("Average input length:", round(np.mean(lengths), 2))

print("\nTokenizer padding ID:", ordering_tokenizer.pad_token_id)
print("Vocabulary size:", ordering_tokenizer.vocab_size)



TRAIN
Valid examples: 121408
Invalid examples: 0
Maximum input length: 512
Average input length: 223.73

VALIDATION
Valid examples: 14845
Invalid examples: 0
Maximum input length: 512
Average input length: 224.65

TEST
Valid examples: 15431
Invalid examples: 0
Maximum input length: 512
Average input length: 223.97

Tokenizer padding ID: 0
Vocabulary size: 250100


In [14]:

import os
import gc
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1. Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# --------------------------------------------------
# 2. GPU memory limit
# --------------------------------------------------

if device.type == "cuda":
    # Limit this process to approximately 1.4 GB
    torch.cuda.set_per_process_memory_fraction(0.35, 0)

    torch.cuda.empty_cache()
    gc.collect()

    print(
        "PyTorch GPU memory limit:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            * 0.35 / (1024 ** 3),
            2
        ),
        "GB"
    )

# --------------------------------------------------
# 3. Convert existing target IDs into labels
# --------------------------------------------------

class OrderingDataset(Dataset):

    def __init__(self, hf_dataset):
        self.dataset = hf_dataset
        self.targets = []

        for example in self.dataset:
            order = extract_order(example["labels"])

            if order is None:
                raise ValueError("Invalid target found!")

            self.targets.append(order)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        example = self.dataset[index]

        return {
            "input_ids": example["input_ids"],
            "labels": self.targets[index]
        }


train_dataset = OrderingDataset(ordering_data["train"])
validation_dataset = OrderingDataset(ordering_data["validation"])
test_dataset = OrderingDataset(ordering_data["test"])

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

# --------------------------------------------------
# 4. Batch padding
# --------------------------------------------------

PAD_ID = ordering_tokenizer.pad_token_id

def collate_fn(batch):

    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:

        ids = item["input_ids"]
        padding_length = max_len - len(ids)

        input_ids.append(
            ids + [PAD_ID] * padding_length
        )

        attention_masks.append(
            [1] * len(ids) + [0] * padding_length
        )

        labels.append(item["labels"])

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.bool),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

print("\nData loaders ready.")


Device: cuda
PyTorch GPU memory limit: 1.4 GB

Dataset sizes:
Train: 121408
Validation: 14845
Test: 15431

Data loaders ready.


In [15]:
import torch.nn as nn
class SentenceOrderingTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        max_length=512,
        num_sentences=5,
        dropout=0.1
    ):
        super().__init__()

        self.d_model = d_model
        self.num_sentences = num_sentences

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        # Positional embeddings for input tokens
        self.position_embedding = nn.Embedding(
            max_length,
            d_model
        )

        # Five learnable output-position queries
        self.query_embeddings = nn.Parameter(
            torch.randn(1, num_sentences, d_model) * 0.02
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.final_norm = nn.LayerNorm(d_model)

        # Each output position predicts one of 5 sentence IDs
        self.classifier = nn.Linear(d_model, num_sentences)

    def forward(self, input_ids, attention_mask):

        batch_size, seq_length = input_ids.shape

        if seq_length > self.position_embedding.num_embeddings:
            raise ValueError(
                f"Input length {seq_length} exceeds maximum length 512"
            )

        # Embed input tokens
        token_embeddings = self.token_embedding(input_ids)

        positions = torch.arange(
            seq_length,
            device=input_ids.device
        ).unsqueeze(0)

        token_embeddings = (
            token_embeddings
            + self.position_embedding(positions)
        )

        # Expand the five query vectors for this batch
        queries = self.query_embeddings.expand(
            batch_size, -1, -1
        )

        # Queries come before input tokens
        combined = torch.cat(
            [queries, token_embeddings],
            dim=1
        )

        # Query positions are never masked.
        # Padding tokens in the input are masked.
        query_mask = torch.zeros(
            batch_size,
            self.num_sentences,
            dtype=torch.bool,
            device=input_ids.device
        )

        padding_mask = ~attention_mask

        full_padding_mask = torch.cat(
            [query_mask, padding_mask],
            dim=1
        )

        # Transformer processing
        encoded = self.transformer(
            combined,
            src_key_padding_mask=full_padding_mask
        )

        # Keep the five query outputs
        query_outputs = encoded[:, :self.num_sentences, :]

        query_outputs = self.final_norm(query_outputs)

        # Shape: batch_size x 5 x 5
        logits = self.classifier(query_outputs)

        return logits


# --------------------------------------------------
# 5. Initialize model
# --------------------------------------------------

model = SentenceOrderingTransformer(
    vocab_size=ordering_tokenizer.vocab_size
).to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model initialized successfully.")
print("Total parameters:", f"{total_parameters:,}")
print("Model device:", next(model.parameters()).device)


C:\Users\HOME\AppData\Local\Temp\ipykernel_25540\3181879921.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Model initialized successfully.
Total parameters: 32,344,837
Model device: cuda:0


In [13]:
# --------------------------------------------------
# Load the saved Epoch 1 checkpoint
# --------------------------------------------------

checkpoint_path = (
    MODEL_DIR
    / "sentence_ordering_transformer"
    / "model_state_dict.pt"
)

print("Checkpoint path:", checkpoint_path)
print("Checkpoint exists:", checkpoint_path.exists())

assert checkpoint_path.exists(), (
    "Saved Model 2 checkpoint was not found. "
    "Do not continue training."
)

# Load saved model weights
saved_state = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(saved_state)

print("\nEpoch 1 checkpoint loaded successfully!")
print("Model device:", next(model.parameters()).device)
print("Checkpoint size (MB):",
      round(checkpoint_path.stat().st_size / (1024 * 1024), 2))

Checkpoint path: E:\punctuation_project\models\sentence_ordering_transformer\model_state_dict.pt
Checkpoint exists: True

Epoch 1 checkpoint loaded successfully!
Model device: cuda:0
Checkpoint size (MB): 123.4


In [7]:

import torch.nn.functional as F

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)

# Get one batch
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Reset GPU memory measurement
if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# One training step
model.train()
optimizer.zero_grad(set_to_none=True)

if device.type == "cuda":
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        logits = model(input_ids, attention_mask)

        loss = F.cross_entropy(
            logits.reshape(-1, 5),
            labels.reshape(-1)
        )
else:
    logits = model(input_ids, attention_mask)

    loss = F.cross_entropy(
        logits.reshape(-1, 5),
        labels.reshape(-1)
    )

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

optimizer.step()

print("Smoke test successful!")
print("Input shape:", tuple(input_ids.shape))
print("Output shape:", tuple(logits.shape))
print("Target shape:", tuple(labels.shape))
print("Loss:", round(loss.item(), 4))

if device.type == "cuda":
    peak_memory = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
    )

    print("Peak allocated GPU memory:", round(peak_memory, 3), "GB")
    print(
        "Peak reserved GPU memory:",
        round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3),
        "GB"
    )


Smoke test successful!
Input shape: (4, 225)
Output shape: (4, 5, 5)
Target shape: (4, 5)
Loss: 1.4784
Peak allocated GPU memory: 0.618 GB
Peak reserved GPU memory: 0.637 GB


In [16]:

import json
import itertools
from tqdm.auto import tqdm
from torch.amp import GradScaler, autocast

# --------------------------------------------------
# 1. Start with a fresh model
# --------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

model = SentenceOrderingTransformer(
    vocab_size=ordering_tokenizer.vocab_size
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)

scaler = GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

# --------------------------------------------------
# 2. Prepare permutation decoding
# --------------------------------------------------

all_permutations = torch.tensor(
    list(itertools.permutations(range(5))),
    dtype=torch.long,
    device=device
)

# --------------------------------------------------
# 3. Validation function
# --------------------------------------------------

def evaluate_ordering(model, loader):

    model.eval()

    total_examples = 0
    correct_positions = 0
    exact_matches = 0
    raw_correct_positions = 0

    with torch.no_grad():

        for batch in tqdm(loader, desc="Validation"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(input_ids, attention_mask)

            # Independent predictions, before permutation constraint
            raw_predictions = logits.argmax(dim=-1)

            raw_correct_positions += (
                raw_predictions == labels
            ).sum().item()

            # Log probabilities
            log_probs = torch.log_softmax(
                logits.float(),
                dim=-1
            )

            batch_size = input_ids.size(0)

            # Score all 120 possible permutations.
            # This guarantees each sentence ID appears exactly once.
            candidate_scores = (
                log_probs[:, None, :, :]
                .expand(-1, 120, -1, -1)
                .gather(
                    3,
                    all_permutations[None, :, :, None].expand(
                        batch_size, -1, -1, 1
                    )
                )
                .squeeze(-1)
                .sum(dim=-1)
            )

            best_indices = candidate_scores.argmax(dim=1)

            predictions = all_permutations[best_indices]

            correct_positions += (
                predictions == labels
            ).sum().item()

            exact_matches += (
                predictions == labels
            ).all(dim=1).sum().item()

            total_examples += batch_size

    return {
        "position_accuracy": correct_positions / (total_examples * 5),
        "exact_accuracy": exact_matches / total_examples,
        "raw_position_accuracy": raw_correct_positions / (total_examples * 5)
    }


# --------------------------------------------------
# 4. Train for one complete epoch
# --------------------------------------------------

EPOCHS = 1

best_exact_accuracy = -1.0

save_dir = MODEL_DIR / "sentence_ordering_transformer"
save_dir.mkdir(parents=True, exist_ok=True)

if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0
    processed_batches = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(input_ids, attention_mask)

            loss = F.cross_entropy(
                logits.reshape(-1, 5),
                labels.reshape(-1)
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        processed_batches += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss / processed_batches:.4f}"
        )

    average_train_loss = total_loss / processed_batches

    print(f"\nEpoch {epoch + 1} training loss: {average_train_loss:.4f}")

    # --------------------------------------------------
    # 5. Validate
    # --------------------------------------------------

    metrics = evaluate_ordering(model, validation_loader)

    print("\nVALIDATION RESULTS")
    print("Raw position accuracy:",
          f"{metrics['raw_position_accuracy'] * 100:.2f}%")

    print("Position accuracy (unique IDs):",
          f"{metrics['position_accuracy'] * 100:.2f}%")

    print("Exact permutation accuracy:",
          f"{metrics['exact_accuracy'] * 100:.2f}%")

    # --------------------------------------------------
    # 6. Save best model
    # --------------------------------------------------

    if metrics["exact_accuracy"] > best_exact_accuracy:

        best_exact_accuracy = metrics["exact_accuracy"]

        torch.save(
            model.state_dict(),
            save_dir / "model_state_dict.pt"
        )

        ordering_tokenizer.save_pretrained(str(save_dir))

        config = {
            "vocab_size": ordering_tokenizer.vocab_size,
            "d_model": 128,
            "num_heads": 4,
            "num_layers": 2,
            "ff_dim": 256,
            "max_length": 512,
            "num_sentences": 5,
            "pad_token_id": PAD_ID
        }

        with open(save_dir / "config.json", "w", encoding="utf-8") as f:
            json.dump(config, f, indent=4)

        print("\nBest model saved to:", save_dir)

# --------------------------------------------------
# 7. GPU memory report
# --------------------------------------------------

if device.type == "cuda":

    print("\nGPU MEMORY REPORT")

    print(
        "Peak allocated:",
        round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
        "GB"
    )

    print(
        "Peak reserved:",
        round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3),
        "GB"
    )

print("\nMODEL 2 TRAINING FINISHED")


C:\Users\HOME\AppData\Local\Temp\ipykernel_25540\3181879921.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 1/1:   0%|          | 0/30352 [00:00<?, ?it/s]

NameError: name 'F' is not defined

In [17]:
import torch
import torch.nn.functional as F

checkpoint_path = (
    MODEL_DIR
    / "sentence_ordering_transformer"
    / "model_state_dict.pt"
)

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Best saved model restored.")

Best saved model restored.


In [9]:

import json
import shutil
import gc
from tqdm.auto import tqdm
from torch.amp import autocast

# --------------------------------------------------
# 1. Verify existing training objects
# --------------------------------------------------

assert model is not None
assert optimizer is not None
assert scaler is not None

save_dir = MODEL_DIR / "sentence_ordering_transformer"

checkpoint_path = save_dir / "model_state_dict.pt"

assert checkpoint_path.exists(), "Epoch 1 checkpoint not found!"

# Preserve the original Epoch 1 checkpoint
epoch1_backup = save_dir / "epoch1_model_state_dict.pt"

if not epoch1_backup.exists():
    shutil.copy2(checkpoint_path, epoch1_backup)

print("Epoch 1 checkpoint preserved.")

# --------------------------------------------------
# 2. Training configuration
# --------------------------------------------------

ADDITIONAL_EPOCHS = 5
MIN_IMPROVEMENT = 0.001
EARLY_STOPPING_PATIENCE = 3

# Best validation result from Epoch 1
best_position_accuracy = 0.2057
best_exact_accuracy = 0.0099
best_epoch = 1

epochs_without_improvement = 0

# Reduce learning rate when validation stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

history = []

if device.type == "cuda":
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

# --------------------------------------------------
# 3. Continue training
# --------------------------------------------------

for epoch in range(2, 2 + ADDITIONAL_EPOCHS):

    print("\n" + "=" * 60)
    print(f"STARTING EPOCH {epoch}")
    print("=" * 60)

    model.train()

    total_loss = 0.0
    processed_batches = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}"
    )

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(input_ids, attention_mask)

            loss = F.cross_entropy(
                logits.reshape(-1, 5),
                labels.reshape(-1)
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        processed_batches += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss / processed_batches:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    average_train_loss = total_loss / processed_batches

    print(f"\nEpoch {epoch} training loss: {average_train_loss:.4f}")

    # --------------------------------------------------
    # 4. Validate after each epoch
    # --------------------------------------------------

    metrics = evaluate_ordering(model, validation_loader)

    position_accuracy = metrics["position_accuracy"]
    exact_accuracy = metrics["exact_accuracy"]

    print("\nVALIDATION RESULTS")
    print(
        "Raw position accuracy:",
        f"{metrics['raw_position_accuracy'] * 100:.2f}%"
    )
    print(
        "Position accuracy:",
        f"{position_accuracy * 100:.2f}%"
    )
    print(
        "Exact permutation accuracy:",
        f"{exact_accuracy * 100:.2f}%"
    )

    # --------------------------------------------------
    # 5. Save only when validation improves meaningfully
    # --------------------------------------------------

    improved = (
        position_accuracy >
        best_position_accuracy + MIN_IMPROVEMENT
    )

    if improved:

        best_position_accuracy = position_accuracy
        best_exact_accuracy = exact_accuracy
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            save_dir / "model_state_dict.pt"
        )

        # Save optimizer and scaler states for recovery
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_position_accuracy": best_position_accuracy,
                "best_exact_accuracy": best_exact_accuracy
            },
            save_dir / "training_checkpoint.pt"
        )

        print("\nImproved model saved!")

    else:

        epochs_without_improvement += 1

        print(
            "\nNo meaningful validation improvement.",
            f"Patience: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    # Scheduler monitors position accuracy
    scheduler.step(position_accuracy)

    # Record results
    history.append({
        "epoch": epoch,
        "train_loss": average_train_loss,
        "raw_position_accuracy": metrics["raw_position_accuracy"],
        "position_accuracy": position_accuracy,
        "exact_accuracy": exact_accuracy,
        "learning_rate": optimizer.param_groups[0]["lr"]
    })

    with open(save_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=4)

    print("Current learning rate:", optimizer.param_groups[0]["lr"])

    # Early stopping
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping: validation has stopped improving.")
        break

# --------------------------------------------------
# 6. Restore best validation checkpoint
# --------------------------------------------------

best_state = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(best_state)

print("\n" + "=" * 60)
print("TRAINING FINISHED")
print("=" * 60)

print("Best epoch:", best_epoch)
print(
    "Best validation position accuracy:",
    f"{best_position_accuracy * 100:.2f}%"
)
print(
    "Best validation exact accuracy:",
    f"{best_exact_accuracy * 100:.2f}%"
)

if device.type == "cuda":
    print(
        "Peak allocated GPU memory:",
        round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
        "GB"
    )

    print(
        "Peak reserved GPU memory:",
        round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3),
        "GB"
    )

print("\nBest model restored into memory.")
print("Model directory:", save_dir)


OSError: [Errno 28] No space left on device

In [14]:
import os
import json
import itertools
import gc

import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from torch.amp import GradScaler, autocast


# ==================================================
# 1. CHECK REQUIRED OBJECTS
# ==================================================

assert model is not None
assert len(train_loader) > 0
assert len(validation_loader) > 0

save_dir = MODEL_DIR / "sentence_ordering_transformer"
save_dir.mkdir(parents=True, exist_ok=True)

best_checkpoint_path = save_dir / "model_state_dict.pt"
latest_checkpoint_path = save_dir / "last_model_state_dict.pt"
history_path = save_dir / "training_history.json"

print("Training examples:", len(train_loader.dataset))
print("Validation examples:", len(validation_loader.dataset))
print("Starting from the loaded Epoch 1 checkpoint.")


# ==================================================
# 2. OPTIMIZER AND MIXED PRECISION
# ==================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

scaler = GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

# All 120 possible permutations of S1, S2, S3, S4, S5
all_permutations = torch.tensor(
    list(itertools.permutations(range(5))),
    dtype=torch.long,
    device=device
)


# ==================================================
# 3. VALIDATION FUNCTION
# ==================================================

def evaluate_ordering(model, loader):

    model.eval()

    total_examples = 0
    correct_positions = 0
    raw_correct_positions = 0
    exact_matches = 0

    with torch.no_grad():

        for batch in tqdm(loader, desc="Validation"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(input_ids, attention_mask)

            # Independent predictions
            raw_predictions = logits.argmax(dim=-1)

            raw_correct_positions += (
                raw_predictions == labels
            ).sum().item()

            # Score every valid permutation
            log_probs = torch.log_softmax(
                logits.float(),
                dim=-1
            )

            batch_size = input_ids.size(0)

            candidate_scores = (
                log_probs[:, None, :, :]
                .expand(-1, 120, -1, -1)
                .gather(
                    3,
                    all_permutations[None, :, :, None].expand(
                        batch_size, -1, -1, 1
                    )
                )
                .squeeze(-1)
                .sum(dim=-1)
            )

            best_indices = candidate_scores.argmax(dim=1)
            predictions = all_permutations[best_indices]

            correct_positions += (
                predictions == labels
            ).sum().item()

            exact_matches += (
                predictions == labels
            ).all(dim=1).sum().item()

            total_examples += batch_size

    return {
        "position_accuracy": correct_positions / (total_examples * 5),
        "exact_accuracy": exact_matches / total_examples,
        "raw_position_accuracy": raw_correct_positions / (total_examples * 5)
    }


# ==================================================
# 4. TRACK THE BEST MODEL
# ==================================================

# Epoch 1 validation results from our previous run
best_exact_accuracy = 0.0099
best_position_accuracy = 0.2057

history = []

EPOCHS = 5


# ==================================================
# 5. CONTINUE TRAINING: EPOCH 2 TO EPOCH 6
# ==================================================

if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

for epoch in range(2, EPOCHS + 2):

    model.train()

    total_loss = 0.0
    processed_batches = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS + 1}"
    )

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits = model(input_ids, attention_mask)

            loss = F.cross_entropy(
                logits.reshape(-1, 5),
                labels.reshape(-1)
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        processed_batches += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss / processed_batches:.4f}"
        )

    average_train_loss = total_loss / processed_batches

    print(f"\nEpoch {epoch} training loss: {average_train_loss:.4f}")


    # ==================================================
    # 6. VALIDATE AFTER EACH EPOCH
    # ==================================================

    metrics = evaluate_ordering(model, validation_loader)

    print("\nVALIDATION RESULTS")
    print(f"Raw position accuracy: {metrics['raw_position_accuracy']:.2%}")
    print(f"Position accuracy (unique IDs): {metrics['position_accuracy']:.2%}")
    print(f"Exact permutation accuracy: {metrics['exact_accuracy']:.2%}")


    # ==================================================
    # 7. SAVE LATEST COMPLETED EPOCH
    # ==================================================

    # Save CPU copies to avoid keeping extra GPU memory occupied.
    state_to_save = {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    }

    temporary_path = save_dir / "last_model_state_dict.tmp"

    torch.save(state_to_save, temporary_path)

    os.replace(
        temporary_path,
        latest_checkpoint_path
    )

    print("Latest checkpoint saved:", latest_checkpoint_path)


    # ==================================================
    # 8. SAVE BEST MODEL IF VALIDATION IMPROVES
    # ==================================================

    is_better = (
        metrics["exact_accuracy"] > best_exact_accuracy
        or (
            metrics["exact_accuracy"] == best_exact_accuracy
            and metrics["position_accuracy"] > best_position_accuracy
        )
    )

    if is_better:

        best_exact_accuracy = metrics["exact_accuracy"]
        best_position_accuracy = metrics["position_accuracy"]

        temporary_path = save_dir / "best_model_state_dict.tmp"

        torch.save(state_to_save, temporary_path)

        os.replace(
            temporary_path,
            best_checkpoint_path
        )

        print("New best model saved!")

    else:
        print("Previous best checkpoint preserved.")


    # ==================================================
    # 9. RECORD TRAINING HISTORY
    # ==================================================

    history.append({
        "epoch": epoch,
        "training_loss": average_train_loss,
        "raw_position_accuracy": metrics["raw_position_accuracy"],
        "position_accuracy": metrics["position_accuracy"],
        "exact_accuracy": metrics["exact_accuracy"],
        "best_exact_accuracy": best_exact_accuracy,
        "best_position_accuracy": best_position_accuracy
    })

    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4)

    print(f"\nCompleted Epoch {epoch}")
    print("-" * 55)

    del state_to_save
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


# ==================================================
# 10. LOAD THE BEST MODEL BACK INTO MEMORY
# ==================================================

best_state = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(best_state)

print("\nTRAINING CONTINUATION FINISHED")
print("Best model:", best_checkpoint_path)
print(f"Best exact permutation accuracy: {best_exact_accuracy:.2%}")
print(f"Best position accuracy: {best_position_accuracy:.2%}")
print("Training history:", history_path)

Training examples: 121408
Validation examples: 14845
Starting from the loaded Epoch 1 checkpoint.


Epoch 2/6:   0%|          | 0/30352 [00:00<?, ?it/s]


Epoch 2 training loss: 1.6093


Validation:   0%|          | 0/3712 [00:00<?, ?it/s]


VALIDATION RESULTS
Raw position accuracy: 20.71%
Position accuracy (unique IDs): 20.55%
Exact permutation accuracy: 0.89%
Latest checkpoint saved: E:\punctuation_project\models\sentence_ordering_transformer\last_model_state_dict.pt
Previous best checkpoint preserved.

Completed Epoch 2
-------------------------------------------------------


Epoch 3/6:   0%|          | 0/30352 [00:00<?, ?it/s]

... (11 outputs left out)

Epoch 6/6:   0%|          | 0/30352 [00:00<?, ?it/s]


Epoch 6 training loss: 1.6056


Validation:   0%|          | 0/3712 [00:00<?, ?it/s]


VALIDATION RESULTS
Raw position accuracy: 21.26%
Position accuracy (unique IDs): 20.79%
Exact permutation accuracy: 1.21%
Latest checkpoint saved: E:\punctuation_project\models\sentence_ordering_transformer\last_model_state_dict.pt
New best model saved!

Completed Epoch 6
-------------------------------------------------------

TRAINING CONTINUATION FINISHED
Best model: E:\punctuation_project\models\sentence_ordering_transformer\model_state_dict.pt
Best exact permutation accuracy: 1.21%
Best position accuracy: 20.79%
Training history: E:\punctuation_project\models\sentence_ordering_transformer\training_history.json


In [3]:
import json
from pathlib import Path

history_path = (
    PROJECT_DIR
    / "models"
    / "sentence_ordering_transformer"
    / "training_history.json"
)

with open(history_path, "r", encoding="utf-8") as f:
    history = json.load(f)

print(json.dumps(history, indent=2))

[
  {
    "epoch": 2,
    "training_loss": 1.6093171293138828,
    "raw_position_accuracy": 0.20712697878073424,
    "position_accuracy": 0.20546985517009095,
    "exact_accuracy": 0.008891882788817784,
    "best_exact_accuracy": 0.0099,
    "best_position_accuracy": 0.2057
  },
  {
    "epoch": 3,
    "training_loss": 1.6082929950374394,
    "raw_position_accuracy": 0.21038733580330077,
    "position_accuracy": 0.21057595149882116,
    "exact_accuracy": 0.010508588750421018,
    "best_exact_accuracy": 0.010508588750421018,
    "best_position_accuracy": 0.21057595149882116
  },
  {
    "epoch": 4,
    "training_loss": 1.6079127527557364,
    "raw_position_accuracy": 0.21387672617042774,
    "position_accuracy": 0.2099023240148198,
    "exact_accuracy": 0.011316941731222633,
    "best_exact_accuracy": 0.011316941731222633,
    "best_position_accuracy": 0.2099023240148198
  },
  {
    "epoch": 5,
    "training_loss": 1.606343754460671,
    "raw_position_accuracy": 0.21120916133378242,
  

In [4]:
from pathlib import Path

save_dir = (
    Path(r"E:\punctuation_project")
    / "models"
    / "sentence_ordering_transformer"
)

for filename in [
    "model_state_dict.pt",
    "last_model_state_dict.pt",
    "training_history.json"
]:
    path = save_dir / filename

    print(
        filename,
        "| Exists:", path.exists(),
        "| Size (MB):",
        round(path.stat().st_size / (1024 * 1024), 2)
        if path.exists() else "Missing"
    )

model_state_dict.pt | Exists: True | Size (MB): 123.4
last_model_state_dict.pt | Exists: True | Size (MB): 123.4
training_history.json | Exists: True | Size (MB): 0.0


In [5]:
from datasets import load_from_disk
from transformers import AutoTokenizer

ordering_data = load_from_disk(str(ORDERING_DATA_PATH))

ordering_tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small",
    use_fast=False
)

for i in range(5):
    example = ordering_data["train"][i]

    print(f"\n{'='*70}")
    print(f"EXAMPLE {i+1}")

    print("\nINPUT:")
    print(
        ordering_tokenizer.decode(
            example["input_ids"],
            skip_special_tokens=True
        )
    )

    print("\nEXPECTED ORDER:")
    print(
        ordering_tokenizer.decode(
            example["labels"],
            skip_special_tokens=True
        )
    )


EXAMPLE 1

INPUT:
restore order: S1: प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी. S2: यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है। S3: यह बिहार प्रान्त में स्थित प्रतिष्ठित अभियांत्रिकी संस्थानों में से एक है जिसमें विभिन्न शाखाएं है। S4: रोहिलखंड विश्वविद्यालय के स्वयं के अभियांत्रिकी प्रवेश परीक्षा के माध्यम से अखिल भारतीय स्तर पर आयोजित किया गया था। S5: इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।

EXPECTED ORDER:
S5 S2 S3 S1 S4

EXAMPLE 2

INPUT:
restore order: S1: बहुत सालो तक यह सोमवार से शनिवार तक ही प्रकाशित होता रहा। S2: यह डेली वर्कर के बाद दूसरा राष्ट्रीय अख़बार था। S3: डेली स्टार ब्रिटन का एक लघु समाचार-पत्र है। S4: लेकिन १५ सितंबर २००२ के बाद यह सनडे (रविवार) को डेली सटार् सनडे (रविवार) के नाम से भी प्रकाशित होने लगा। S5: यह पहली बार २ नोवेंबर १९७८ को प्रकाशित हुआ था।

EXPECTED ORDER:
S3 S5 S2 S1 S4

EXAMPLE 3

INPUT:
restore order: S1: इसके अलावा परमाणु परीक्षणों से वैज्ञानिक,

In [18]:
import torch

model.eval()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    logits = model(input_ids, attention_mask)

print("Input shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Labels shape:", labels.shape)
print("Logits shape:", logits.shape)

print("\nFirst target:", labels[0].tolist())
print(
    "First prediction:",
    logits[0].argmax(dim=-1).tolist()
)

print(
    "\nUnique target IDs:",
    torch.unique(labels).tolist()
)

Input shape: torch.Size([4, 246])
Attention mask shape: torch.Size([4, 246])
Labels shape: torch.Size([4, 5])
Logits shape: torch.Size([4, 5, 5])

First target: [1, 0, 4, 2, 3]
First prediction: [3, 0, 0, 0, 0]

Unique target IDs: [0, 1, 2, 3, 4]


In [19]:
import copy
import torch
import torch.nn.functional as F

# --------------------------------------------------
# 1. Create a separate copy of the current best model
# --------------------------------------------------

diagnostic_model = copy.deepcopy(model).to(device)

diagnostic_optimizer = torch.optim.AdamW(
    diagnostic_model.parameters(),
    lr=1e-3,
    weight_decay=0.0
)

# --------------------------------------------------
# 2. Select ONE fixed batch of four examples
# --------------------------------------------------

fixed_batch = next(iter(train_loader))

input_ids = fixed_batch["input_ids"].to(device)
attention_mask = fixed_batch["attention_mask"].to(device)
labels = fixed_batch["labels"].to(device)

print("Number of examples:", input_ids.size(0))
print("Target labels:")
print(labels.cpu().tolist())

# --------------------------------------------------
# 3. Train repeatedly on these same four examples
# --------------------------------------------------

STEPS = 100

for step in range(1, STEPS + 1):

    diagnostic_model.train()

    diagnostic_optimizer.zero_grad(set_to_none=True)

    logits = diagnostic_model(
        input_ids,
        attention_mask
    )

    loss = F.cross_entropy(
        logits.reshape(-1, 5),
        labels.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        diagnostic_model.parameters(),
        max_norm=1.0
    )

    diagnostic_optimizer.step()

    # Check predictions every 20 steps
    if step % 20 == 0:

        diagnostic_model.eval()

        with torch.no_grad():

            predictions = diagnostic_model(
                input_ids,
                attention_mask
            ).argmax(dim=-1)

            position_accuracy = (
                predictions == labels
            ).float().mean().item()

            exact_accuracy = (
                predictions == labels
            ).all(dim=1).float().mean().item()

        print(f"\nStep {step}")
        print("Loss:", round(loss.item(), 4))
        print(
            "Position accuracy:",
            f"{position_accuracy * 100:.2f}%"
        )
        print(
            "Exact accuracy:",
            f"{exact_accuracy * 100:.2f}%"
        )

print("\nOVERFITTING TEST FINISHED")

Number of examples: 4
Target labels:
[[1, 4, 3, 0, 2], [4, 1, 3, 0, 2], [2, 1, 3, 4, 0], [4, 0, 2, 1, 3]]

Step 20
Loss: 1.0244
Position accuracy: 65.00%
Exact accuracy: 0.00%

Step 40
Loss: 0.459
Position accuracy: 90.00%
Exact accuracy: 50.00%

Step 60
Loss: 0.0837
Position accuracy: 100.00%
Exact accuracy: 100.00%

Step 80
Loss: 0.0261
Position accuracy: 100.00%
Exact accuracy: 100.00%

Step 100
Loss: 0.0227
Position accuracy: 100.00%
Exact accuracy: 100.00%

OVERFITTING TEST FINISHED


In [21]:
from datasets import load_from_disk
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

ordering_processed = load_from_disk(
    str(PROJECT_DIR / "data" / "processed" / "ordering_processed")
)

print(ordering_processed)
print("\nDataset columns:")
print(ordering_processed["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14845
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15431
    })
})

Dataset columns:
['input_ids', 'attention_mask', 'labels']


In [22]:
from transformers import AutoTokenizer

# Load the tokenizer used to create the existing dataset
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

# Take one existing training example
sample = ordering_processed["train"][0]

# Recover its text from the existing token IDs
recovered_text = mt5_tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=True
)

print("RECOVERED INPUT:")
print(recovered_text)

print("\nEXISTING TARGET LABELS:")
print(sample["labels"])

print("\nDATASET COLUMNS:")
print(ordering_processed["train"].column_names)

RECOVERED INPUT:
restore order: S1: प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी. S2: यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है। S3: यह बिहार प्रान्त में स्थित प्रतिष्ठित अभियांत्रिकी संस्थानों में से एक है जिसमें विभिन्न शाखाएं है। S4: रोहिलखंड विश्वविद्यालय के स्वयं के अभियांत्रिकी प्रवेश परीक्षा के माध्यम से अखिल भारतीय स्तर पर आयोजित किया गया था। S5: इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।

EXISTING TARGET LABELS:
[320, 428, 320, 338, 320, 328, 320, 353, 320, 410, 1]

DATASET COLUMNS:
['input_ids', 'attention_mask', 'labels']


In [24]:
target_text = mt5_tokenizer.decode(
    sample["labels"],
    skip_special_tokens=True
)

print("INPUT:")
print(recovered_text)

print("\nTARGET:")
print(target_text)

INPUT:
restore order: S1: प्रारंभ में (१९९५-२००७ तक) प्रवेश एम.जे.पी. S2: यह एम॰जे॰पी॰ रोहिलखंड विश्वविद्यालय के परिसर में स्थित है। S3: यह बिहार प्रान्त में स्थित प्रतिष्ठित अभियांत्रिकी संस्थानों में से एक है जिसमें विभिन्न शाखाएं है। S4: रोहिलखंड विश्वविद्यालय के स्वयं के अभियांत्रिकी प्रवेश परीक्षा के माध्यम से अखिल भारतीय स्तर पर आयोजित किया गया था। S5: इंजीनियरिंग और प्रौद्योगिकी संस्थान, बरेली (आईईटी) उत्तर प्रदेश, बरेली में राज्य सरकार द्वारा वित्त पोषित इंजीनियरिंग कॉलेज है।

TARGET:
S5 S2 S3 S1 S4


In [25]:
import re
from datasets import DatasetDict, load_from_disk
from transformers import AutoTokenizer

# Load IndicBERT tokenizer
indic_tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/indic-bert",
    use_fast=False
)

def convert_batch(batch):
    # Recover original input text and target order
    input_texts = [
        mt5_tokenizer.decode(ids, skip_special_tokens=True)
        for ids in batch["input_ids"]
    ]

    target_texts = [
        mt5_tokenizer.decode(ids, skip_special_tokens=True)
        for ids in batch["labels"]
    ]

    # Convert S1-S5 into numerical labels 0-4
    ordering_labels = []

    for target in target_texts:
        sentence_ids = re.findall(r"S([1-5])", target)

        if len(sentence_ids) != 5 or set(sentence_ids) != set("12345"):
            raise ValueError(f"Invalid target: {target}")

        ordering_labels.append([
            int(sentence_id) - 1
            for sentence_id in sentence_ids
        ])

    # Tokenize inputs using IndicBERT
    encoded = indic_tokenizer(
        input_texts,
        truncation=True,
        max_length=512,
        padding=False
    )

    encoded["labels"] = ordering_labels

    return encoded


# Convert all existing splits
ordering_indicbert = DatasetDict()

for split in ordering_processed.keys():
    print(f"Converting {split}...")

    ordering_indicbert[split] = ordering_processed[split].map(
        convert_batch,
        batched=True,
        batch_size=500,
        remove_columns=ordering_processed[split].column_names
    )

# Save separately
output_path = PROJECT_DIR / "data" / "processed" / "ordering_indicbert_processed"

ordering_indicbert.save_to_disk(str(output_path))

print("\nConversion completed.")
print(ordering_indicbert)
print("\nExample:")
print(ordering_indicbert["train"][0])

Converting train...


Map:   0%|          | 0/121408 [00:00<?, ? examples/s]

Converting validation...


Map:   0%|          | 0/14845 [00:00<?, ? examples/s]

Converting test...


Map:   0%|          | 0/15431 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/121408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14845 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15431 [00:00<?, ? examples/s]


Conversion completed.
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14845
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15431
    })
})

Example:
{'input_ids': [2, 56679, 2740, 216, 98, 1857, 216, 37, 152590, 2092, 20, 92416, 107936, 30, 192638, 9588, 199, 129, 37, 45801, 3878, 9, 1325, 9, 1551, 9, 98, 1603, 216, 114, 3878, 64247, 1325, 64247, 1551, 64247, 2022, 487, 89201, 77636, 1976, 109832, 28689, 1134, 1883, 37, 6511, 2092, 1301, 4090, 229, 4384, 15, 98, 1916, 216, 114, 51092, 37, 339, 229, 2092, 1301, 4090, 229, 5458, 11611, 6676, 229, 33252, 9327, 507, 152133, 96276, 2092, 1301, 45, 4384, 18654, 670, 130, 11204, 60439, 6448, 6169, 1640, 4384, 15, 98, 1938, 216, 2022, 487, 89201, 77636, 1976, 109832, 28689, 1134, 1883, 37680, 1883, 3325

In [27]:
import re
from datasets import DatasetDict

def extract_sentences(input_ids):
    text = mt5_tokenizer.decode(
        input_ids,
        skip_special_tokens=True
    )

    matches = re.findall(
        r"(?:^|\s)S([1-5]):\s*(.*?)(?=\sS[1-5]:|$)",
        text,
        flags=re.DOTALL
    )

    if len(matches) != 5:
        return None

    if sorted(int(x[0]) for x in matches) != [1, 2, 3, 4, 5]:
        return None

    sentences = [sentence.strip() for _, sentence in matches]

    if any(not sentence for sentence in sentences):
        return None

    return sentences


def prepare_sentence_batch(batch):
    all_sentence_ids = []
    all_sentence_masks = []
    all_labels = []

    for input_ids, label_ids in zip(
        batch["input_ids"], batch["labels"]
    ):
        sentences = extract_sentences(input_ids)

        if sentences is None:
            raise ValueError("Incomplete example passed filtering.")

        target = mt5_tokenizer.decode(
            label_ids,
            skip_special_tokens=True
        )

        order = [
            int(x) - 1
            for x in re.findall(r"S([1-5])", target)
        ]

        if len(order) != 5 or sorted(order) != [0, 1, 2, 3, 4]:
            raise ValueError(f"Invalid target: {target}")

        encoded = indic_tokenizer(
            sentences,
            truncation=True,
            max_length=128,
            padding="max_length"
        )

        all_sentence_ids.append(encoded["input_ids"])
        all_sentence_masks.append(encoded["attention_mask"])
        all_labels.append(order)

    return {
        "sentence_input_ids": all_sentence_ids,
        "sentence_attention_mask": all_sentence_masks,
        "labels": all_labels
    }


ordering_sentence_data = DatasetDict()

for split in ordering_processed.keys():
    original = ordering_processed[split]

    # Remove only examples where all five sentences cannot be recovered
    valid = original.filter(
        lambda example: extract_sentences(example["input_ids"]) is not None
    )

    print(
        f"{split}: retained {len(valid):,} / {len(original):,}; "
        f"removed {len(original) - len(valid):,}"
    )

    ordering_sentence_data[split] = valid.map(
        prepare_sentence_batch,
        batched=True,
        batch_size=200,
        remove_columns=valid.column_names
    )

sentence_path = (
    PROJECT_DIR / "data" / "processed"
    / "ordering_indicbert_sentence_processed"
)

ordering_sentence_data.save_to_disk(str(sentence_path))

print("\nFinal dataset:")
print(ordering_sentence_data)
print("\nExample labels:", ordering_sentence_data["train"][0]["labels"])

Filter:   0%|          | 0/121408 [00:00<?, ? examples/s]

train: retained 121,316 / 121,408; removed 92


Map:   0%|          | 0/121316 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14845 [00:00<?, ? examples/s]

validation: retained 14,835 / 14,845; removed 10


Map:   0%|          | 0/14835 [00:00<?, ? examples/s]

... (3 outputs left out)

Saving the dataset (0/3 shards):   0%|          | 0/121316 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14835 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15417 [00:00<?, ? examples/s]


Final dataset:
DatasetDict({
    train: Dataset({
        features: ['labels', 'sentence_input_ids', 'sentence_attention_mask'],
        num_rows: 121316
    })
    validation: Dataset({
        features: ['labels', 'sentence_input_ids', 'sentence_attention_mask'],
        num_rows: 14835
    })
    test: Dataset({
        features: ['labels', 'sentence_input_ids', 'sentence_attention_mask'],
        num_rows: 15417
    })
})

Example labels: [4, 1, 2, 0, 3]


In [28]:
import torch
import torch.nn as nn
import math

from transformers import AutoModel
from torch.utils.data import DataLoader


# 1. Load pretrained IndicBERT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = AutoModel.from_pretrained("ai4bharat/indic-bert")
encoder = encoder.to(device)
encoder.eval()

# Freeze pretrained weights
for param in encoder.parameters():
    param.requires_grad = False

HIDDEN_SIZE = encoder.config.hidden_size

print("Hidden size:", HIDDEN_SIZE)
print("Encoder parameters:", sum(p.numel() for p in encoder.parameters()))


# 2. Custom sentence-ordering network
class SentenceOrderingHead(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.query_embeddings = nn.Parameter(
            torch.randn(1, 5, hidden_size) * 0.02
        )

        self.position_embeddings = nn.Parameter(
            torch.randn(1, 5, hidden_size) * 0.02
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=8,
            batch_first=True
        )

        self.norm = nn.LayerNorm(hidden_size)

        self.query_projection = nn.Linear(hidden_size, hidden_size)
        self.key_projection = nn.Linear(hidden_size, hidden_size)

    def forward(self, sentence_embeddings):

        batch_size = sentence_embeddings.size(0)

        # Identify each sentence's shuffled input position
        keys = sentence_embeddings + self.position_embeddings

        queries = self.query_embeddings.expand(batch_size, -1, -1)

        attended, _ = self.cross_attention(
            queries,
            keys,
            keys
        )

        attended = self.norm(queries + attended)

        q = self.query_projection(attended)
        k = self.key_projection(keys)

        # Output: batch_size x 5 output positions x 5 input sentences
        logits = torch.matmul(q, k.transpose(1, 2))
        logits = logits / math.sqrt(q.size(-1))

        return logits


ordering_head = SentenceOrderingHead(HIDDEN_SIZE).to(device)

print("\nCustom head parameters:",
      sum(p.numel() for p in ordering_head.parameters()))

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden size: 768
Encoder parameters: 33443584

Custom head parameters: 3552768


In [29]:
# One example, five sentences
sample = ordering_sentence_data["train"][0]

input_ids = torch.tensor(
    sample["sentence_input_ids"],
    dtype=torch.long
).unsqueeze(0).to(device)

attention_mask = torch.tensor(
    sample["sentence_attention_mask"],
    dtype=torch.long
).unsqueeze(0).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# Encode five sentences using frozen IndicBERT
with torch.no_grad():
    batch_size, num_sentences, seq_length = input_ids.shape

    outputs = encoder(
        input_ids=input_ids.reshape(-1, seq_length),
        attention_mask=attention_mask.reshape(-1, seq_length)
    )

    # Use the CLS representation of each sentence
    sentence_embeddings = outputs.last_hidden_state[:, 0, :]

    sentence_embeddings = sentence_embeddings.reshape(
        batch_size, num_sentences, HIDDEN_SIZE
    )

# Predict sentence order
logits = ordering_head(sentence_embeddings)

print("Input shape:", input_ids.shape)
print("Sentence embeddings:", sentence_embeddings.shape)
print("Output logits:", logits.shape)

if torch.cuda.is_available():
    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 3),
        "GB"
    )

Input shape: torch.Size([1, 5, 128])
Sentence embeddings: torch.Size([1, 5, 768])
Output logits: torch.Size([1, 5, 5])
Peak GPU memory: 0.776 GB


In [30]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from itertools import permutations
from pathlib import Path
import json
import time

# Configuration
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 1e-3

# Keep IndicBERT frozen
encoder.eval()

# Only the ordering head is trained
optimizer = torch.optim.AdamW(
    ordering_head.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()

def collate_ordering(batch):
    return {
        "input_ids": torch.tensor(
            [x["sentence_input_ids"] for x in batch],
            dtype=torch.long
        ),
        "attention_mask": torch.tensor(
            [x["sentence_attention_mask"] for x in batch],
            dtype=torch.long
        ),
        "labels": torch.tensor(
            [x["labels"] for x in batch],
            dtype=torch.long
        )
    }

train_loader = DataLoader(
    ordering_sentence_data["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_ordering
)

validation_loader = DataLoader(
    ordering_sentence_data["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_ordering
)

# Evaluate using the best valid permutation
all_permutations = torch.tensor(
    list(permutations(range(5))),
    device=device
)

def evaluate_ordering():
    ordering_head.eval()

    correct_positions = 0
    correct_sequences = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for batch in validation_loader:

            ids = batch["input_ids"].to(device)
            masks = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            b, n, length = ids.shape

            outputs = encoder(
                input_ids=ids.reshape(-1, length),
                attention_mask=masks.reshape(-1, length)
            )

            embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings = embeddings.reshape(b, n, HIDDEN_SIZE)

            logits = ordering_head(embeddings)

            loss = criterion(
                logits.reshape(-1, 5),
                labels.reshape(-1)
            )

            total_loss += loss.item() * b

            # Raw position accuracy
            predictions = logits.argmax(dim=-1)

            # Constrained permutation decoding
            scores = logits[:, None, :, :].expand(
                -1, 120, -1, -1
            )

            permutation_scores = scores.gather(
                3,
                all_permutations[None, :, :, None].expand(
                    b, -1, -1, -1
                )
            ).squeeze(-1).sum(dim=-1)

            best_indices = permutation_scores.argmax(dim=1)

            constrained_predictions = all_permutations[best_indices]

            correct_positions += (
                constrained_predictions == labels
            ).sum().item()

            correct_sequences += (
                constrained_predictions == labels
            ).all(dim=1).sum().item()

            total += b

    return {
        "loss": total_loss / total,
        "position_accuracy": correct_positions / (total * 5),
        "exact_accuracy": correct_sequences / total
    }


# Training
best_exact = -1

save_dir = Path(
    r"E:\punctuation_project\models\sentence_ordering_indicbert"
)
save_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):

    ordering_head.train()
    start_time = time.time()
    total_loss = 0

    for step, batch in enumerate(train_loader, 1):

        ids = batch["input_ids"].to(device)
        masks = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        b, n, length = ids.shape

        # Frozen encoder: no gradients
        with torch.no_grad():
            outputs = encoder(
                input_ids=ids.reshape(-1, length),
                attention_mask=masks.reshape(-1, length)
            )

            embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings = embeddings.reshape(
                b, n, HIDDEN_SIZE
            )

        logits = ordering_head(embeddings)

        loss = criterion(
            logits.reshape(-1, 5),
            labels.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if step % 2000 == 0:
            print(
                f"Epoch {epoch+1} | Step {step}/{len(train_loader)} "
                f"| Loss: {total_loss / step:.4f}"
            )

    metrics = evaluate_ordering()

    print(f"\nEpoch {epoch+1} completed in {(time.time()-start_time)/60:.1f} min")
    print(f"Training loss: {total_loss/len(train_loader):.4f}")
    print(f"Validation loss: {metrics['loss']:.4f}")
    print(f"Position accuracy: {metrics['position_accuracy']*100:.2f}%")
    print(f"Exact accuracy: {metrics['exact_accuracy']*100:.2f}%")

    if metrics["exact_accuracy"] > best_exact:
        best_exact = metrics["exact_accuracy"]

        torch.save({
            "ordering_head": ordering_head.state_dict(),
            "hidden_size": HIDDEN_SIZE,
            "epoch": epoch + 1,
            "metrics": metrics
        }, save_dir / "best_model.pt")

        print("Best checkpoint saved.")

    print("-" * 50)

Epoch 1 | Step 2000/30329 | Loss: 1.6099
Epoch 1 | Step 4000/30329 | Loss: 1.6098
Epoch 1 | Step 6000/30329 | Loss: 1.6098
Epoch 1 | Step 8000/30329 | Loss: 1.6097
Epoch 1 | Step 10000/30329 | Loss: 1.6097
Epoch 1 | Step 12000/30329 | Loss: 1.6097
Epoch 1 | Step 14000/30329 | Loss: 1.6097
Epoch 1 | Step 16000/30329 | Loss: 1.6096
Epoch 1 | Step 18000/30329 | Loss: 1.6096
Epoch 1 | Step 20000/30329 | Loss: 1.6096
Epoch 1 | Step 22000/30329 | Loss: 1.6096
Epoch 1 | Step 24000/30329 | Loss: 1.6096
Epoch 1 | Step 26000/30329 | Loss: 1.6096
Epoch 1 | Step 28000/30329 | Loss: 1.6096
Epoch 1 | Step 30000/30329 | Loss: 1.6096

Epoch 1 completed in 91.5 min
Training loss: 1.6096
Validation loss: 1.6094
Position accuracy: 20.63%
Exact accuracy: 0.79%
Best checkpoint saved.
--------------------------------------------------
Epoch 2 | Step 2000/30329 | Loss: 1.6095
Epoch 2 | Step 4000/30329 | Loss: 1.6094
Epoch 2 | Step 6000/30329 | Loss: 1.6095
Epoch 2 | Step 8000/30329 | Loss: 1.6095
Epoch 2 | S

KeyboardInterrupt: 

In [1]:

from datasets import load_from_disk

DATASET_PATH = r"E:\punctuation_project\data\processed\ordering_indicbert_sentence_processed"

dataset = load_from_disk(DATASET_PATH)

print("Dataset splits:")
for split in dataset:
    print(f"{split}: {len(dataset[split])} examples")

print("\nTraining dataset columns:")
print(dataset["train"].column_names)

print("\nFirst training example:")
example = dataset["train"][0]

for key, value in example.items():
    print(f"\n{key}:")
    print(value if len(str(value)) < 1500 else str(value)[:1500] + "...")


Dataset splits:
train: 121316 examples
validation: 14835 examples
test: 15417 examples

Training dataset columns:
['labels', 'sentence_input_ids', 'sentence_attention_mask']

First training example:

labels:
[4, 1, 2, 0, 3]

sentence_input_ids:
[[2, 37, 152590, 2092, 20, 92416, 107936, 30, 192638, 9588, 199, 129, 37, 45801, 3878, 9, 1325, 9, 1551, 9, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 114, 3878, 64247, 1325, 64247, 1551, 64247, 2022, 487, 89201, 77636, 1976, 109832, 28689, 1134, 1883, 37, 6511, 2092, 1301, 4090, 229, 4384, 15, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [2]:

from transformers import AutoTokenizer

# Load the tokenizer used for the sentence-ordering dataset.
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

def decode_example(example, example_number):
    print("\n" + "=" * 80)
    print(f"EXAMPLE {example_number}")
    print("=" * 80)

    sentences = []

    for i, (ids, label) in enumerate(
        zip(example["sentence_input_ids"], example["labels"])
    ):
        sentence = tokenizer.decode(
            [token_id for token_id in ids if token_id != 0],
            skip_special_tokens=True
        )

        sentences.append({
            "input_position": i,
            "original_position": int(label),
            "text": sentence
        })

        print(f"\nInput sentence S{i + 1}")
        print(f"Label: {label}")
        print(f"Text: {sentence}")

    print("\n" + "-" * 80)
    print("SENTENCES SORTED BY LABEL")
    print("-" * 80)

    for item in sorted(sentences, key=lambda x: x["original_position"]):
        print(f"\nPosition {item['original_position'] + 1}: {item['text']}")


for i in range(3):
    decode_example(dataset["train"][i], i + 1)



EXAMPLE 1

Input sentence S1
Label: 4
Text: पररभ म (१९९५-२००७ तक) परवश एम.ज.प.

Input sentence S2
Label: 1
Text: यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।

Input sentence S3
Label: 2
Text: यह बहर परनत म सथत परतषठत अभयतरक ससथन म स एक ह जसम वभनन शखए ह।

Input sentence S4
Label: 0
Text: रहलखड वशववदयलय क सवय क अभयतरक परवश परकष क मधयम स अखल भरतय सतर पर आयजत कय गय थ।

Input sentence S5
Label: 3
Text: इजनयरग और परदयगक ससथन, बरल (आईईट) उततर परदश, बरल म रजय सरकर दवर वतत पषत इजनयरग कलज ह।

--------------------------------------------------------------------------------
SENTENCES SORTED BY LABEL
--------------------------------------------------------------------------------

Position 1: रहलखड वशववदयलय क सवय क अभयतरक परवश परकष क मधयम स अखल भरतय सतर पर आयजत कय गय थ।

Position 2: यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।

Position 3: यह बहर परनत म सथत परतषठत अभयतरक ससथन म स एक ह जसम वभनन शखए ह।

Position 4: इजनयरग और परदयगक ससथन, बरल (आईईट) उततर परदश, बरल म रजय सरकर दवर वतत पषत इजनयरग कलज ह।

Posi

In [3]:

from collections import Counter

label_counts = Counter()

for example in dataset["train"]:
    label_counts.update(example["labels"])

print("Label distribution:")
for label in sorted(label_counts):
    print(f"Position {label}: {label_counts[label]}")

print("\nExpected labels per example: 5")
print("Labels in first example:", dataset["train"][0]["labels"])
print("Are all five positions present?",
      sorted(dataset["train"][0]["labels"]) == [0, 1, 2, 3, 4])


Label distribution:
Position 0: 121316
Position 1: 121316
Position 2: 121316
Position 3: 121316
Position 4: 121316

Expected labels per example: 5
Labels in first example: [4, 1, 2, 0, 3]
Are all five positions present? True


In [10]:

import json
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

keywords = [
    "permutation",
    "shuffle",
    "random.shuffle",
    "ordering_processed",
    "sentence_order",
    "sentence_input_ids",
    "original_order"
]

found = 0

for notebook_path in PROJECT_DIR.rglob("*.ipynb"):

    # Skip checkpoint and environment folders
    if any(part.startswith(".") or part in {
        "venv", ".venv", "__pycache__"
    } for part in notebook_path.parts):
        continue

    try:
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        for cell_number, cell in enumerate(notebook.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))
            matches = [
                keyword for keyword in keywords
                if keyword.lower() in source.lower()
            ]

            if matches:
                found += 1
                print("\n" + "=" * 80)
                print("NOTEBOOK:", notebook_path.name)
                print("CELL:", cell_number)
                print("MATCHES:", matches)
                print("=" * 80)
                print(source[:5000])

    except Exception as e:
        print("Could not read:", notebook_path.name, e)

print("\nTotal matching cells:", found)



NOTEBOOK: 02_sentence_ordering.ipynb
CELL: 0
MATCHES: ['ordering_processed']
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"

ORDERING_DATA_PATH = DATA_DIR / "ordering_processed"

NOTEBOOK: 02_sentence_ordering.ipynb
CELL: 2
MATCHES: ['sentence_order']
print("Project exists:", PROJECT_DIR.exists())
print("Ordering dataset exists:", ORDERING_DATA_PATH.exists())
print("Model checkpoint exists:",
      (MODEL_DIR / "sentence_ordering_transformer" / "model_state_dict.pt").exists())

NOTEBOOK: 02_sentence_ordering.ipynb
CELL: 3
MATCHES: ['ordering_processed']
from pathlib import Path
from datasets import load_from_disk

PROJECT_DIR = Path(r"E:\punctuation_project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

ORDERING_PATH = PROCESSED_DIR / "ordering_processed"

print("Dataset path:", ORDERING_PATH)
print("Dataset exists:", ORDERING_PATH.exists())

ordering_data = load_from_disk(

In [5]:

import json
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

keywords = [
    "def clean_hindi",
    "def split_sentences",
    "source_dataset =",
    "generate_ordering_examples"
]

for notebook_path in PROJECT_DIR.rglob("*.ipynb"):

    try:
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        for cell_number, cell in enumerate(notebook.get("cells", [])):

            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            if any(k in source for k in keywords):

                print("\n" + "=" * 80)
                print("NOTEBOOK:", notebook_path.name)
                print("CELL:", cell_number)
                print("=" * 80)
                print(source[:8000])

    except Exception as e:
        print("Could not read:", notebook_path.name, e)



NOTEBOOK: 02_sentence_ordering.ipynb
CELL: 32

import json
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

keywords = [
    "def clean_hindi",
    "def split_sentences",
    "source_dataset =",
    "generate_ordering_examples"
]

for notebook_path in PROJECT_DIR.rglob("*.ipynb"):

    try:
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        for cell_number, cell in enumerate(notebook.get("cells", [])):

            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            if any(k in source for k in keywords):

                print("\n" + "=" * 80)
                print("NOTEBOOK:", notebook_path.name)
                print("CELL:", cell_number)
                print("=" * 80)
                print(source[:8000])

    except Exception as e:
        print("Could not read:", notebook_path.name, e)

Could not read: 04_end_to_end_pipeline.i

In [6]:

from itertools import combinations
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

example = dataset["train"][0]

sentences = []

for ids in example["sentence_input_ids"]:
    # Remove padding tokens
    ids = [x for x in ids if x != tokenizer.pad_token_id]

    text = tokenizer.decode(
        ids,
        skip_special_tokens=True
    )

    sentences.append(text)

labels = example["labels"]

print("Original training example:")
print("Labels:", labels)

print("\nGENERATED PAIRWISE COMPARISONS")
print("=" * 80)

for i, j in combinations(range(5), 2):

    # 1 means sentence i comes before sentence j
    # 0 means sentence j comes before sentence i
    pair_label = int(labels[i] < labels[j])

    if pair_label == 1:
        first, second = i, j
    else:
        first, second = j, i

    print(f"\nComparison: S{i+1} vs S{j+1}")
    print("Sentence A:", sentences[i])
    print("Sentence B:", sentences[j])
    print("Prediction target:", pair_label)
    print(f"Correct order: S{first+1} -> S{second+1}")

print("\nTotal comparisons:", len(list(combinations(range(5), 2))))


Original training example:
Labels: [4, 1, 2, 0, 3]

GENERATED PAIRWISE COMPARISONS

Comparison: S1 vs S2
Sentence A: पररभ म (१९९५-२००७ तक) परवश एम.ज.प.
Sentence B: यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।
Prediction target: 0
Correct order: S2 -> S1

Comparison: S1 vs S3
Sentence A: पररभ म (१९९५-२००७ तक) परवश एम.ज.प.
Sentence B: यह बहर परनत म सथत परतषठत अभयतरक ससथन म स एक ह जसम वभनन शखए ह।
Prediction target: 0
Correct order: S3 -> S1

Comparison: S1 vs S4
Sentence A: पररभ म (१९९५-२००७ तक) परवश एम.ज.प.
Sentence B: रहलखड वशववदयलय क सवय क अभयतरक परवश परकष क मधयम स अखल भरतय सतर पर आयजत कय गय थ।
Prediction target: 0
Correct order: S4 -> S1

Comparison: S1 vs S5
Sentence A: पररभ म (१९९५-२००७ तक) परवश एम.ज.प.
Sentence B: इजनयरग और परदयगक ससथन, बरल (आईईट) उततर परदश, बरल म रजय सरकर दवर वतत पषत इजनयरग कलज ह।
Prediction target: 0
Correct order: S5 -> S1

Comparison: S2 vs S3
Sentence A: यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।
Sentence B: यह बहर परनत म सथत परतषठत अभयतरक ससथन म स एक ह जसम वभनन श

In [7]:

from datasets import load_from_disk
from transformers import AutoTokenizer

DATASET_PATH = r"E:\punctuation_project\data\processed\ordering_indicbert_sentence_processed"

dataset = load_from_disk(DATASET_PATH)

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

example = dataset["train"][0]

print("LABELS:", example["labels"])

for i in range(5):

    ids = example["sentence_input_ids"][i]
    mask = example["sentence_attention_mask"][i]

    # Keep only actual tokens, excluding padding
    valid_ids = [
        token_id
        for token_id, attention in zip(ids, mask)
        if attention == 1
    ]

    decoded = tokenizer.decode(
        valid_ids,
        skip_special_tokens=True
    )

    print("\n" + "=" * 80)
    print(f"SENTENCE {i+1}")
    print("Token count:", len(valid_ids))
    print("Token IDs:", valid_ids[:30])
    print("Decoded text:", decoded)


LABELS: [4, 1, 2, 0, 3]

SENTENCE 1
Token count: 21
Token IDs: [2, 37, 152590, 2092, 20, 92416, 107936, 30, 192638, 9588, 199, 129, 37, 45801, 3878, 9, 1325, 9, 1551, 9, 3]
Decoded text: पररभ म (१९९५-२००७ तक) परवश एम.ज.प.

SENTENCE 2
Token count: 26
Token IDs: [2, 114, 3878, 64247, 1325, 64247, 1551, 64247, 2022, 487, 89201, 77636, 1976, 109832, 28689, 1134, 1883, 37, 6511, 2092, 1301, 4090, 229, 4384, 15, 3]
Decoded text: यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।

SENTENCE 3
Token count: 34
Token IDs: [2, 114, 51092, 37, 339, 229, 2092, 1301, 4090, 229, 5458, 11611, 6676, 229, 33252, 9327, 507, 152133, 96276, 2092, 1301, 45, 4384, 18654, 670, 130, 11204, 60439, 6448, 6169]
Decoded text: यह बहर परनत म सथत परतषठत अभयतरक ससथन म स एक ह जसम वभनन शखए ह।

SENTENCE 4
Token count: 41
Token IDs: [2, 2022, 487, 89201, 77636, 1976, 109832, 28689, 1134, 1883, 37680, 1883, 33252, 9327, 507, 37, 45801, 37, 507, 11611, 1883, 36159, 44395, 1301, 50743, 487, 23269, 1134, 20252, 367]
Decoded text: रहलखड

In [8]:

# Inspect the original Wikipedia article used for ordering data

example = dataset["train"][0]

print("CURRENT PROCESSED SENTENCES")
print("=" * 80)

for i, ids in enumerate(example["sentence_input_ids"]):
    mask = example["sentence_attention_mask"][i]

    valid_ids = [
        token_id
        for token_id, attention in zip(ids, mask)
        if attention == 1
    ]

    print(f"\nSentence {i+1}:")
    print(tokenizer.decode(valid_ids, skip_special_tokens=True))


print("\n\nORIGINAL WIKIPEDIA ARTICLE")
print("=" * 80)

# Search the original article dataset for a matching phrase.
# Use a distinctive word sequence from the decoded sentence.
search_phrase = "रहलखड"

matches = []

for article in wiki_splits["train"]:
    if search_phrase in article["text"]:
        matches.append(article["text"])
        if len(matches) == 3:
            break

print("Matching articles:", len(matches))

for i, article_text in enumerate(matches):
    print(f"\nARTICLE {i+1}:")
    print(article_text[:3000])


CURRENT PROCESSED SENTENCES

Sentence 1:
पररभ म (१९९५-२००७ तक) परवश एम.ज.प.

Sentence 2:
यह एम॰ज॰प॰ रहलखड वशववदयलय क परसर म सथत ह।

Sentence 3:
यह बहर परनत म सथत परतषठत अभयतरक ससथन म स एक ह जसम वभनन शखए ह।

Sentence 4:
रहलखड वशववदयलय क सवय क अभयतरक परवश परकष क मधयम स अखल भरतय सतर पर आयजत कय गय थ।

Sentence 5:
इजनयरग और परदयगक ससथन, बरल (आईईट) उततर परदश, बरल म रजय सरकर दवर वतत पषत इजनयरग कलज ह।


ORIGINAL WIKIPEDIA ARTICLE


NameError: name 'wiki_splits' is not defined

In [12]:

from datasets import load_dataset, DatasetDict

# 1. Load the Hindi Wikipedia dataset
wiki_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.hi",
    split="train"
)

# 2. Recreate the original train/validation/test split
wiki_temp = wiki_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

wiki_remaining = wiki_temp["test"].train_test_split(
    test_size=0.50,
    seed=42
)

wiki_splits = DatasetDict({
    "train": wiki_temp["train"],
    "validation": wiki_remaining["train"],
    "test": wiki_remaining["test"]
})

# 3. Verify
print("Dataset splits created successfully!")
print("Train:", len(wiki_splits["train"]))
print("Validation:", len(wiki_splits["validation"]))
print("Test:", len(wiki_splits["test"]))


Dataset splits created successfully!
Train: 130474
Validation: 16309
Test: 16310


In [14]:

import re
import unicodedata

def clean_hindi(text):
    """Clean Hindi Wikipedia text while preserving Devanagari content."""
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_sentences(text):
    """Split Hindi text into sentences."""
    sentences = re.split(
        r"(?<=[।॥!?])\s+|(?<=\.)\s+",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip().split()) >= 3
    ]


print("Preprocessing functions are ready.")

# Quick verification
sample = "यह पहला वाक्य है। यह दूसरा वाक्य है।"
print(split_sentences(clean_hindi(sample)))


Preprocessing functions are ready.
['यह पहला वाक्य है।', 'यह दूसरा वाक्य है।']


In [15]:

import random
from itertools import combinations
from pathlib import Path

from datasets import Dataset, DatasetDict, Features, Sequence, Value

# Existing wiki_splits and split_sentences from your previous cell
# Load tokenizer if it is not already available.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

MAX_LENGTH = 128
MAX_PAIRS_PER_PARAGRAPH = 20
SEED = 42

def generate_pairwise_examples(source_dataset):

    rng = random.Random(SEED)

    for article in source_dataset:

        cleaned_article = clean_hindi(article["text"])

        for paragraph in cleaned_article.split("\n"):

            sentences = split_sentences(paragraph)

            if len(sentences) < 2:
                continue

            pairs = list(combinations(range(len(sentences)), 2))

            # Limit pairs to control dataset size
            if len(pairs) > MAX_PAIRS_PER_PARAGRAPH:
                pairs = rng.sample(pairs, MAX_PAIRS_PER_PARAGRAPH)

            for i, j in pairs:

                # Randomly swap sentence order
                if rng.random() < 0.5:
                    sentence_a = sentences[i]
                    sentence_b = sentences[j]
                    label = 1
                else:
                    sentence_a = sentences[j]
                    sentence_b = sentences[i]
                    label = 0

                encoded = tokenizer(
                    sentence_a,
                    sentence_b,
                    truncation=True,
                    max_length=MAX_LENGTH,
                    padding="max_length"
                )

                yield {
                    "input_ids": encoded["input_ids"],
                    "attention_mask": encoded["attention_mask"],
                    "labels": label
                }


pairwise_features = Features({
    "input_ids": Sequence(Value("int32")),
    "attention_mask": Sequence(Value("int8")),
    "labels": Value("int64")
})


def build_pairwise_dataset(source, split_name):

    return Dataset.from_generator(
        generate_pairwise_examples,
        gen_kwargs={"source_dataset": source},
        features=pairwise_features,
        split=split_name,
        keep_in_memory=False,
        writer_batch_size=500
    )


# Build all three splits separately
pairwise_data = DatasetDict({
    "train": build_pairwise_dataset(
        wiki_splits["train"], "train"
    ),
    "validation": build_pairwise_dataset(
        wiki_splits["validation"], "validation"
    ),
    "test": build_pairwise_dataset(
        wiki_splits["test"], "test"
    )
})

# Save to a NEW directory. Existing data remains untouched.
SAVE_PATH = Path(
    r"E:\punctuation_project\data\processed\ordering_pairwise_indicbert"
)

pairwise_data.save_to_disk(str(SAVE_PATH))

print("PAIRWISE DATASET CREATED")
for split in pairwise_data:
    print(split, len(pairwise_data[split]))

print("\nFirst example:")
print("Labels:", pairwise_data["train"][0]["labels"])
print("Token count:", len(pairwise_data["train"][0]["input_ids"]))


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/1164572 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/145973 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/144116 [00:00<?, ? examples/s]

PAIRWISE DATASET CREATED
train 1164572
validation 145973
test 144116

First example:
Labels: 0
Token count: 128


In [18]:

# Verify pairwise label logic with known sentence positions

sentences = [
    "Sentence A",
    "Sentence B",
    "Sentence C",
    "Sentence D"
]

test_cases = [
    (0, 1, 1),  # A before B
    (1, 0, 0),  # B before A
    (0, 3, 1),  # A before D
    (3, 0, 0),  # D before A
    (2, 1, 0),  # C after B
]

for a, b, expected in test_cases:
    label = int(a < b)
    assert label == expected

print("All label logic checks passed!")


All label logic checks passed!


In [16]:

# Inspect one pairwise example
example = pairwise_data["train"][0]

print("Keys:", example.keys())
print("Label:", example["labels"])
print("Input IDs shape:", len(example["input_ids"]))
print("Attention mask shape:", len(example["attention_mask"]))

# Decode the input to inspect the two sentences
print("\nDecoded input:")
print(tokenizer.decode(
    example["input_ids"],
    skip_special_tokens=False
))


Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Label: 0
Input IDs shape: 128
Attention mask shape: 128

Decoded input:
[CLS] सनदरभ बलमण अमम मलयलम कवय सगरह मलयलम सहतय[SEP] सतर हदयम (मलयलम: സതരഹദയ) भरत स मलयलम भष क सरवधक परतभवन कवयतरय म स एक नलपत बलमण अमम क कवय सगरह ह, ज 1939 म पहल बर मलयलम भष म परकशत हआ।[SEP]<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


In [17]:

# Inspect the first pair directly from the original dataset
example = pairwise_data["train"][0]

decoded = tokenizer.decode(
    example["input_ids"],
    skip_special_tokens=True
)

print("Decoded pair:")
print(decoded)

print("\nLabel:", example["labels"])
print("Expected meaning:",
      "B comes before A" if example["labels"] == 0
      else "A comes before B")


Decoded pair:
सनदरभ बलमण अमम मलयलम कवय सगरह मलयलम सहतय सतर हदयम (मलयलम: സതരഹദയ) भरत स मलयलम भष क सरवधक परतभवन कवयतरय म स एक नलपत बलमण अमम क कवय सगरह ह, ज 1939 म पहल बर मलयलम भष म परकशत हआ।

Label: 0
Expected meaning: B comes before A


In [19]:

import random

rng = random.Random(42)
checked = 0
errors = []

for article in wiki_splits["train"]:
    cleaned = clean_hindi(article["text"])

    for paragraph in cleaned.split("\n"):
        sentences = split_sentences(paragraph)

        if len(sentences) < 2:
            continue

        pairs = list(combinations(range(len(sentences)), 2))

        if len(pairs) > MAX_PAIRS_PER_PARAGRAPH:
            pairs = rng.sample(pairs, MAX_PAIRS_PER_PARAGRAPH)

        for i, j in pairs:
            # Reproduce the generator's random swap
            if rng.random() < 0.5:
                sentence_a, sentence_b = sentences[i], sentences[j]
                label = 1
            else:
                sentence_a, sentence_b = sentences[j], sentences[i]
                label = 0

            # Verify against original sentence positions
            original_a = sentences.index(sentence_a)
            original_b = sentences.index(sentence_b)
            expected_label = int(original_a < original_b)

            if label != expected_label:
                errors.append({
                    "A": sentence_a,
                    "B": sentence_b,
                    "label": label,
                    "expected": expected_label
                })

            checked += 1

            if checked >= 100:
                break

        if checked >= 100:
            break

    if checked >= 100:
        break

print("Examples checked:", checked)
print("Incorrect labels:", len(errors))

if errors:
    print("\nFirst error:", errors[0])
else:
    print("\nAll 100 labels match their original sentence order.")


Examples checked: 100
Incorrect labels: 0

All 100 labels match their original sentence order.


In [20]:

import os
import math
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from pathlib import Path

# -----------------------------
# 1. Configuration
# -----------------------------

SEED = 42
BATCH_SIZE = 8
GRAD_ACCUMULATION = 2
EPOCHS = 1
LEARNING_RATE = 2e-5

TRAIN_LIMIT = 50_000
VAL_LIMIT = 5_000

MODEL_NAME = "ai4bharat/indic-bert"

SAVE_DIR = Path(
    r"E:\punctuation_project\models\sentence_ordering_pairwise_indicbert"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = SAVE_DIR / "baseline.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

print("Device:", device)
print("Mixed precision:", USE_AMP)

# -----------------------------
# 2. Select manageable subsets
# -----------------------------

train_subset = (
    pairwise_data["train"]
    .shuffle(seed=SEED)
    .select(range(min(TRAIN_LIMIT, len(pairwise_data["train"]))))
)

val_subset = (
    pairwise_data["validation"]
    .shuffle(seed=SEED)
    .select(range(min(VAL_LIMIT, len(pairwise_data["validation"]))))
)

train_subset = train_subset.with_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_subset = val_subset.with_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Training examples:", len(train_subset))
print("Validation examples:", len(val_subset))

# -----------------------------
# 3. Define pairwise model
# -----------------------------

class PairwiseOrderingModel(nn.Module):

    def __init__(self, model_name):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Use the [CLS] representation
        cls_output = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(
            self.dropout(cls_output)
        )

        return logits


model = PairwiseOrderingModel(MODEL_NAME).to(device)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

# -----------------------------
# 4. Optimizer and scheduler
# -----------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

total_steps = math.ceil(
    len(train_loader) / GRAD_ACCUMULATION
) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

criterion = nn.CrossEntropyLoss()

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# -----------------------------
# 5. Training
# -----------------------------

if USE_AMP:
    torch.cuda.reset_peak_memory_stats()

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast(enabled=USE_AMP):

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            scaled_loss = loss / GRAD_ACCUMULATION

        scaler.scale(scaled_loss).backward()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        should_update = (
            (step + 1) % GRAD_ACCUMULATION == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if (step + 1) % 100 == 0:
            progress.set_postfix(
                loss=f"{total_loss / (step + 1):.4f}",
                accuracy=f"{100 * correct / total:.2f}%"
            )

    train_loss = total_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # -----------------------------
    # 6. Validation
    # -----------------------------

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch in tqdm(val_loader, desc="Validation"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=USE_AMP):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                loss = criterion(logits, labels)

            val_loss += loss.item()

            predictions = logits.argmax(dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_loss /= len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    print(f"\nEpoch {epoch + 1} Results")
    print(f"Train Loss:      {train_loss:.4f}")
    print(f"Train Accuracy:  {train_accuracy:.2f}%")
    print(f"Val Loss:        {val_loss:.4f}")
    print(f"Val Accuracy:    {val_accuracy:.2f}%")

# -----------------------------
# 7. GPU memory measurement
# -----------------------------

if USE_AMP:

    peak_memory = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
    )

    print(f"\nPeak GPU memory: {peak_memory:.3f} GB")

else:
    peak_memory = None
    print("\nGPU unavailable: memory limit not measured.")

# -----------------------------
# 8. Save baseline checkpoint
# -----------------------------

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "peak_gpu_memory_gb": peak_memory,
        "max_length": 128
    },
    CHECKPOINT_PATH
)

print("\nBaseline checkpoint saved:")
print(CHECKPOINT_PATH)


Device: cuda
Mixed precision: True
Training examples: 50000
Validation examples: 5000


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 33445122


C:\Users\HOME\AppData\Local\Temp\ipykernel_28004\2616581905.py:155: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


Epoch 1/1:   0%|          | 0/6250 [00:00<?, ?it/s]

C:\Users\HOME\AppData\Local\Temp\ipykernel_28004\2616581905.py:186: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Validation:   0%|          | 0/625 [00:00<?, ?it/s]

C:\Users\HOME\AppData\Local\Temp\ipykernel_28004\2616581905.py:251: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):



Epoch 1 Results
Train Loss:      0.6931
Train Accuracy:  50.17%
Val Loss:        0.6935
Val Accuracy:    48.30%

Peak GPU memory: 0.907 GB

Baseline checkpoint saved:
E:\punctuation_project\models\sentence_ordering_pairwise_indicbert\baseline.pt


In [21]:

import json

# Save tokenizer
tokenizer.save_pretrained(SAVE_DIR)

# Save model configuration and training metadata
metadata = {
    "model_name": MODEL_NAME,
    "architecture": "IndicBERT + binary classification head",
    "max_length": MAX_LENGTH if "MAX_LENGTH" in globals() else 128,
    "num_labels": 2,
    "label_mapping": {
        "0": "Sentence B comes before sentence A",
        "1": "Sentence A comes before sentence B"
    },
    "train_accuracy": train_accuracy,
    "validation_accuracy": val_accuracy,
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "peak_gpu_memory_gb": peak_memory,
    "status": "Experimental baseline; not ready for reliable inference"
}

with open(SAVE_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("MODEL ARTIFACTS SAVED")
for file in SAVE_DIR.iterdir():
    print(file.name)


MODEL ARTIFACTS SAVED
baseline.pt
metadata.json
tokenizer.json
tokenizer_config.json


In [22]:

from collections import Counter
import torch

# Check label distribution
labels = pairwise_data["train"]["labels"]

print("Training label distribution:")
print(Counter(labels))

# Check validation label distribution
val_labels = pairwise_data["validation"]["labels"]

print("\nValidation label distribution:")
print(Counter(val_labels))

# Check model prediction distribution
model.eval()

predictions = []
actual_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        actual_labels.extend(labels.cpu().tolist())

print("\nPrediction distribution:")
print(Counter(predictions))

print("\nActual validation distribution:")
print(Counter(actual_labels))


Training label distribution:
Counter({0: 582713, 1: 581859})

Validation label distribution:
Counter({0: 73094, 1: 72879})

Prediction distribution:
Counter({0: 5000})

Actual validation distribution:
Counter({1: 2585, 0: 2415})


In [23]:

# Inspect prediction probabilities and parameter updates

model.eval()

batch = next(iter(val_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    logits = model(input_ids, attention_mask)
    probabilities = torch.softmax(logits, dim=1)

print("First 10 prediction probabilities:")
print(probabilities[:10].cpu())

print("\nFirst 10 actual labels:")
print(labels[:10].cpu())

print("\nLogit statistics:")
print("Class 0 mean:", logits[:, 0].mean().item())
print("Class 1 mean:", logits[:, 1].mean().item())
print("Class 0 std:", logits[:, 0].std().item())
print("Class 1 std:", logits[:, 1].std().item())

# Check whether encoder parameters are trainable
trainable = sum(
    p.numel() for p in model.encoder.parameters()
    if p.requires_grad
)

total = sum(
    p.numel() for p in model.encoder.parameters()
)

print("\nEncoder parameters:")
print("Trainable:", trainable)
print("Total:", total)


First 10 prediction probabilities:
tensor([[0.5041, 0.4959],
        [0.5044, 0.4956],
        [0.5044, 0.4956],
        [0.5041, 0.4959],
        [0.5043, 0.4957],
        [0.5043, 0.4957],
        [0.5044, 0.4956],
        [0.5041, 0.4959]])

First 10 actual labels:
tensor([0, 0, 1, 0, 1, 0, 1, 1])

Logit statistics:
Class 0 mean: -0.04847211763262749
Class 1 mean: -0.06548142433166504
Class 0 std: 0.00026571855414658785
Class 1 std: 0.00032749047386460006

Encoder parameters:
Trainable: 33443584
Total: 33443584


In [24]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel
from tqdm.auto import tqdm

# 1. Select a tiny dataset
tiny_data = (
    pairwise_data["train"]
    .shuffle(seed=42)
    .select(range(256))
    .with_format(
        "torch",
        columns=["input_ids", "attention_mask", "labels"]
    )
)

tiny_loader = DataLoader(
    tiny_data,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

# 2. Create a completely fresh model
diagnostic_model = PairwiseOrderingModel(
    "ai4bharat/indic-bert"
).to(device)

optimizer_diag = torch.optim.AdamW(
    diagnostic_model.parameters(),
    lr=2e-5,
    weight_decay=0.0
)

criterion_diag = nn.CrossEntropyLoss()

# 3. Train repeatedly on the same 256 examples
DIAGNOSTIC_EPOCHS = 15

for epoch in range(DIAGNOSTIC_EPOCHS):

    diagnostic_model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in tiny_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_diag.zero_grad(set_to_none=True)

        logits = diagnostic_model(
            input_ids,
            attention_mask
        )

        loss = criterion_diag(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            diagnostic_model.parameters(),
            1.0
        )

        optimizer_diag.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {total_loss / len(tiny_loader):.4f} | "
        f"Accuracy: {100 * correct / total:.2f}%"
    )

# 4. Save separately
torch.save(
    diagnostic_model.state_dict(),
    SAVE_DIR / "overfit_diagnostic.pt"
)

print("\nDiagnostic model saved.")


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 01 | Loss: 0.6929 | Accuracy: 50.78%
Epoch 02 | Loss: 0.6943 | Accuracy: 50.39%
Epoch 03 | Loss: 0.6922 | Accuracy: 51.17%
Epoch 04 | Loss: 0.6851 | Accuracy: 61.72%
Epoch 05 | Loss: 0.6625 | Accuracy: 64.06%
Epoch 06 | Loss: 0.5991 | Accuracy: 75.00%
Epoch 07 | Loss: 0.4966 | Accuracy: 85.16%
Epoch 08 | Loss: 0.3991 | Accuracy: 91.41%
Epoch 09 | Loss: 0.4524 | Accuracy: 90.23%
Epoch 10 | Loss: 0.1943 | Accuracy: 94.92%
Epoch 11 | Loss: 0.1820 | Accuracy: 94.92%
Epoch 12 | Loss: 0.0362 | Accuracy: 98.83%
Epoch 13 | Loss: 0.0267 | Accuracy: 99.22%
Epoch 14 | Loss: 0.0240 | Accuracy: 99.61%
Epoch 15 | Loss: 0.0020 | Accuracy: 100.00%

Diagnostic model saved.


In [25]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel
from tqdm.auto import tqdm
from pathlib import Path

# Fresh diagnostic dataset
fp32_train = (
    pairwise_data["train"]
    .shuffle(seed=123)
    .select(range(5000))
    .with_format("torch", columns=[
        "input_ids", "attention_mask", "labels"
    ])
)

fp32_val = (
    pairwise_data["validation"]
    .shuffle(seed=123)
    .select(range(1000))
    .with_format("torch", columns=[
        "input_ids", "attention_mask", "labels"
    ])
)

train_loader_fp32 = DataLoader(
    fp32_train, batch_size=4, shuffle=True
)

val_loader_fp32 = DataLoader(
    fp32_val, batch_size=4, shuffle=False
)

# Fresh model
fp32_model = PairwiseOrderingModel(
    "ai4bharat/indic-bert"
).to(device)

optimizer_fp32 = torch.optim.AdamW(
    fp32_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

criterion_fp32 = nn.CrossEntropyLoss()

ACCUMULATION = 4
EPOCHS = 3

for epoch in range(EPOCHS):

    fp32_model.train()
    total_loss = 0
    correct = 0
    total = 0

    optimizer_fp32.zero_grad(set_to_none=True)

    for step, batch in enumerate(
        tqdm(train_loader_fp32, desc=f"FP32 Epoch {epoch+1}")
    ):

        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Full precision: no autocast and no GradScaler
        logits = fp32_model(ids, mask)
        loss = criterion_fp32(logits, labels)

        (loss / ACCUMULATION).backward()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

        if (
            (step + 1) % ACCUMULATION == 0
            or (step + 1) == len(train_loader_fp32)
        ):
            torch.nn.utils.clip_grad_norm_(
                fp32_model.parameters(), 1.0
            )
            optimizer_fp32.step()
            optimizer_fp32.zero_grad(set_to_none=True)

    train_accuracy = 100 * correct / total

    # Validation
    fp32_model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader_fp32:

            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = fp32_model(ids, mask)
            loss = criterion_fp32(logits, labels)

            val_loss += loss.item()
            val_correct += (
                logits.argmax(1) == labels
            ).sum().item()
            val_total += labels.size(0)

    print(f"\nEpoch {epoch+1}")
    print(f"Train loss: {total_loss / len(train_loader_fp32):.4f}")
    print(f"Train accuracy: {train_accuracy:.2f}%")
    print(f"Validation loss: {val_loss / len(val_loader_fp32):.4f}")
    print(f"Validation accuracy: {100 * val_correct / val_total:.2f}%")

# Save separately
torch.save(
    fp32_model.state_dict(),
    SAVE_DIR / "fp32_diagnostic.pt"
)

print("\nFP32 diagnostic checkpoint saved.")


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FP32 Epoch 1:   0%|          | 0/1250 [00:00<?, ?it/s]


Epoch 1
Train loss: 0.6937
Train accuracy: 49.74%
Validation loss: 0.6925
Validation accuracy: 51.70%


FP32 Epoch 2:   0%|          | 0/1250 [00:00<?, ?it/s]


Epoch 2
Train loss: 0.6933
Train accuracy: 49.76%
Validation loss: 0.6927
Validation accuracy: 51.70%


FP32 Epoch 3:   0%|          | 0/1250 [00:00<?, ?it/s]


Epoch 3
Train loss: 0.6939
Train accuracy: 49.40%
Validation loss: 0.6932
Validation accuracy: 48.30%

FP32 diagnostic checkpoint saved.


In [26]:

import re
import unicodedata

def clean_hindi(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)

    # Normalize line endings but preserve paragraph boundaries
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Collapse spaces and tabs WITHOUT removing newlines
    text = re.sub(r"[^\S\n]+", " ", text)

    # Normalize excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Inspect an original article
sample_article = clean_hindi(wiki_splits["train"][0]["text"])

paragraphs = [
    p.strip()
    for p in sample_article.split("\n")
    if p.strip()
]

print("Paragraphs found:", len(paragraphs))

for i, paragraph in enumerate(paragraphs[:3]):
    sentences = split_sentences(paragraph)

    print(f"\nParagraph {i+1}")
    print("Sentence count:", len(sentences))
    print("First sentence:", sentences[0] if sentences else "None")


Paragraphs found: 5

Paragraph 1
Sentence count: 1
First sentence: स्त्री हृदयम (मलयालम: സ്ത്രീഹൃദയം) भारत से मलयालम भाषा की सर्वाधिक प्रतिभावान कवयित्रियों में से एक नालापत बालमणि अम्मा का काव्य संग्रह है, जो 1939 में पहली बार मलयालम भाषा में प्रकाशित हुआ।

Paragraph 2
Sentence count: 0
First sentence: None

Paragraph 3
Sentence count: 0
First sentence: None


In [27]:

raw_article = wiki_splits["train"][0]["text"]

cleaned_article = clean_hindi(raw_article)

for i, paragraph in enumerate(cleaned_article.split("\n")):
    paragraph = paragraph.strip()

    if not paragraph:
        continue

    sentences = split_sentences(paragraph)

    print(f"\n--- Paragraph {i+1} ---")
    print("Characters:", len(paragraph))
    print("Sentences detected:", len(sentences))
    print("Text preview:", paragraph[:400])




--- Paragraph 1 ---
Characters: 190
Sentences detected: 1
Text preview: स्त्री हृदयम (मलयालम: സ്ത്രീഹൃദയം) भारत से मलयालम भाषा की सर्वाधिक प्रतिभावान कवयित्रियों में से एक नालापत बालमणि अम्मा का काव्य संग्रह है, जो 1939 में पहली बार मलयालम भाषा में प्रकाशित हुआ।

--- Paragraph 3 ---
Characters: 7
Sentences detected: 0
Text preview: सन्दर्भ

--- Paragraph 5 ---
Characters: 12
Sentences detected: 0
Text preview: बालमणि अम्मा

--- Paragraph 6 ---
Characters: 19
Sentences detected: 1
Text preview: मलयालम काव्य संग्रह

--- Paragraph 7 ---
Characters: 14
Sentences detected: 0
Text preview: मलयालम साहित्य


In [28]:

test_encoding = tokenizer(
    "भारत एक विशाल देश है।",
    "इसकी राजधानी नई दिल्ली है।",
    truncation=True,
    max_length=128,
    padding="max_length"
)

print("Tokenizer output keys:", test_encoding.keys())

print("\nToken type IDs:")
if "token_type_ids" in test_encoding:
    print(test_encoding["token_type_ids"][:25])
else:
    print("Not provided")

print("\nModel configuration:")
print("Model type:", model.encoder.config.model_type)
print("Type vocabulary size:", model.encoder.config.type_vocab_size)

print("\nCurrent dataset keys:")
print(pairwise_data["train"].column_names)


Tokenizer output keys: KeysView({'input_ids': [2, 23269, 45, 130, 103308, 45114, 4384, 15, 3, 70, 507, 34886, 18023, 1042, 2637, 487, 4384, 15, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]})

Token type IDs:
Not provided

Model configuration:
Model type: albert
Type vocabulary size: 2

Current dataset keys:
['

In [29]:

import torch

encoding = tokenizer(
    "भारत एक विशाल देश है।",
    "इसकी राजधानी नई दिल्ली है।",
    truncation=True,
    max_length=128,
    padding="max_length"
)

input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]

sep_id = tokenizer.sep_token_id
pad_id = tokenizer.pad_token_id

# Find the separator tokens belonging to the two sentences.
sep_positions = [
    i for i, token_id in enumerate(input_ids)
    if token_id == sep_id
]

print("SEP token ID:", sep_id)
print("PAD token ID:", pad_id)
print("SEP positions:", sep_positions)

assert len(sep_positions) >= 2, "Expected two SEP tokens."

# Construct token_type_ids.
token_type_ids = [0] * len(input_ids)

first_sep = sep_positions[0]
second_sep = sep_positions[1]

for i in range(first_sep + 1, second_sep + 1):
    token_type_ids[i] = 1

# Padding remains segment 0.
for i, mask in enumerate(attention_mask):
    if mask == 0:
        token_type_ids[i] = 0

print("\nFirst 25 token IDs:")
print(input_ids[:25])

print("\nFirst 25 token type IDs:")
print(token_type_ids[:25])

print("\nFirst 25 attention mask values:")
print(attention_mask[:25])


SEP token ID: 3
PAD token ID: 0
SEP positions: [8, 18]

First 25 token IDs:
[2, 23269, 45, 130, 103308, 45114, 4384, 15, 3, 70, 507, 34886, 18023, 1042, 2637, 487, 4384, 15, 3, 0, 0, 0, 0, 0, 0]

First 25 token type IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

First 25 attention mask values:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]


In [30]:

import random
import re
import unicodedata

from itertools import combinations
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Configuration
SEED = 42
MAX_LENGTH = 128
MAX_PAIRS_PER_PARAGRAPH = 20

SAVE_DIR_V2 = Path(
    r"E:\punctuation_project\data\processed\ordering_pairwise_indicbert_v2"
)

SAVE_DIR_V2.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")


# Preserve paragraph boundaries.
def clean_hindi(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def generate_pairwise_examples(source_dataset):
    rng = random.Random(SEED)

    for article in source_dataset:
        cleaned_article = clean_hindi(article["text"])

        for paragraph in cleaned_article.split("\n"):
            paragraph = paragraph.strip()

            if not paragraph:
                continue

            sentences = split_sentences(paragraph)

            if len(sentences) < 2:
                continue

            pairs = list(combinations(range(len(sentences)), 2))

            if len(pairs) > MAX_PAIRS_PER_PARAGRAPH:
                pairs = rng.sample(pairs, MAX_PAIRS_PER_PARAGRAPH)

            for i, j in pairs:

                # Label 1: A comes before B.
                # Label 0: B comes before A.
                if rng.random() < 0.5:
                    sentence_a = sentences[i]
                    sentence_b = sentences[j]
                    label = 1
                else:
                    sentence_a = sentences[j]
                    sentence_b = sentences[i]
                    label = 0

                encoded = tokenizer(
                    sentence_a,
                    sentence_b,
                    truncation=True,
                    max_length=MAX_LENGTH,
                    padding="max_length"
                )

                input_ids = encoded["input_ids"]
                attention_mask = encoded["attention_mask"]

                # Construct ALBERT segment IDs.
                sep_positions = [
                    idx for idx, token_id in enumerate(input_ids)
                    if token_id == tokenizer.sep_token_id
                ]

                if len(sep_positions) < 2:
                    continue

                first_sep = sep_positions[0]
                second_sep = sep_positions[1]

                token_type_ids = [0] * len(input_ids)

                for idx in range(first_sep + 1, second_sep + 1):
                    token_type_ids[idx] = 1

                yield {
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "token_type_ids": token_type_ids,
                    "labels": label
                }


# Generate all three splits.
pairwise_data_v2 = DatasetDict()

for split_name in ["train", "validation", "test"]:

    print(f"Generating {split_name}...")

    generated = list(
        generate_pairwise_examples(wiki_splits[split_name])
    )

    pairwise_data_v2[split_name] = Dataset.from_list(generated)

    print(
        f"{split_name}: "
        f"{len(pairwise_data_v2[split_name]):,} examples"
    )

# Save without touching the previous dataset.
pairwise_data_v2.save_to_disk(str(SAVE_DIR_V2))

print("\nDataset saved:", SAVE_DIR_V2)
print("Columns:", pairwise_data_v2["train"].column_names)


Generating train...
train: 2,229,585 examples
Generating validation...
validation: 276,242 examples
Generating test...
test: 283,189 examples


Saving the dataset (0/4 shards):   0%|          | 0/2229585 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/276242 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/283189 [00:00<?, ? examples/s]


Dataset saved: E:\punctuation_project\data\processed\ordering_pairwise_indicbert_v2
Columns: ['input_ids', 'attention_mask', 'token_type_ids', 'labels']


In [31]:

from collections import Counter

sample = pairwise_data_v2["train"][0]

print("Dataset sizes:")
for split in pairwise_data_v2:
    print(split, len(pairwise_data_v2[split]))

print("\nColumns:", pairwise_data_v2["train"].column_names)

print("\nSequence lengths:")
for key in ["input_ids", "attention_mask", "token_type_ids"]:
    print(key, len(sample[key]))

print("\nToken type distribution:")
print(Counter(sample["token_type_ids"]))

print("\nLabel distribution:")
print(Counter(pairwise_data_v2["train"]["labels"]))

assert "token_type_ids" in pairwise_data_v2["train"].column_names
assert len(sample["input_ids"]) == MAX_LENGTH
assert len(sample["token_type_ids"]) == MAX_LENGTH
assert set(sample["token_type_ids"]).issubset({0, 1})

print("\nBasic dataset checks passed.")


Dataset sizes:
train 2229585
validation 276242
test 283189

Columns: ['input_ids', 'attention_mask', 'token_type_ids', 'labels']

Sequence lengths:
input_ids 128
attention_mask 128
token_type_ids 128

Token type distribution:
Counter({0: 104, 1: 24})

Label distribution:
Counter({1: 1115694, 0: 1113891})

Basic dataset checks passed.


In [33]:

from transformers import AlbertForPreTraining
import torch

# CPU inference to avoid exceeding the GPU memory limit.
sop_model = AlbertForPreTraining.from_pretrained(
    "ai4bharat/indic-bert"
)

sop_model.eval()
sop_model.to("cpu")

print("Model loaded successfully.")
print("Model architecture:", type(sop_model).__name__)
print("SOP classifier:", sop_model.sop_classifier)

print("\nSOP output layer:")
print(sop_model.sop_classifier)


Loading weights:   0%|          | 0/34 [00:00<?, ?it/s]

Model loaded successfully.
Model architecture: AlbertForPreTraining
SOP classifier: AlbertSOPHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=2, bias=True)
)

SOP output layer:
AlbertSOPHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=2, bias=True)
)


In [34]:

from torch.utils.data import DataLoader
from collections import Counter
import torch

# Use a small sample to keep CPU inference manageable.
test_subset = (
    pairwise_data_v2["test"]
    .shuffle(seed=123)
    .select(range(500))
)

test_subset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

test_loader = DataLoader(
    test_subset,
    batch_size=8,
    shuffle=False
)

correct = 0
total = 0
prediction_counts = Counter()

with torch.no_grad():

    for batch in test_loader:

        outputs = sop_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"]
        )

        # ALBERT SOP class 0 represents original order;
        # class 1 represents swapped order.
        predictions = outputs.sop_logits.argmax(dim=1)

        # Our dataset: label 1 = A before B.
        # SOP: class 0 = A before B.
        expected = 1 - batch["labels"]

        correct += (predictions == expected).sum().item()
        total += len(expected)

        prediction_counts.update(predictions.tolist())

print("Examples:", total)
print("SOP accuracy:", round(correct / total, 4))
print("Prediction distribution:", prediction_counts)


Examples: 500
SOP accuracy: 0.52
Prediction distribution: Counter({0: 496, 1: 4})


In [35]:

from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

print("PROJECT DIRECTORY")
print(PROJECT_DIR)

print("\nTOP-LEVEL CONTENTS")

for item in sorted(PROJECT_DIR.iterdir()):
    print(
        "[DIR] " if item.is_dir() else "[FILE]",
        item.name
    )

print("\nMODEL CHECKPOINT CONTENTS")

model_dirs = [
    PROJECT_DIR / "models" / "indicbert_punctuation_weighted_final",
    PROJECT_DIR / "models" / "sentence_ordering_indicbert",
]

for directory in model_dirs:
    print(f"\n{directory}")

    if directory.exists():
        for item in sorted(directory.iterdir()):
            print("  ", item.name)
    else:
        print("  Directory not found")


PROJECT DIRECTORY
E:\punctuation_project

TOP-LEVEL CONTENTS
[DIR]  .venv
[FILE] 02_sentence_ordering.ipynb
[FILE] 03_translation.ipynb
[FILE] 04_end_to_end_pipeline.ipynb
[FILE] Add-Punctuation-and-Translate.ipynb
[DIR]  data
[DIR]  models

MODEL CHECKPOINT CONTENTS

E:\punctuation_project\models\indicbert_punctuation_weighted_final
   config.json
   model.safetensors
   tokenizer.json
   tokenizer_config.json
   training_args.bin

E:\punctuation_project\models\sentence_ordering_indicbert
   best_model.pt


In [36]:

import json
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

notebooks = [
    "02_sentence_ordering.ipynb",
    "03_translation.ipynb",
    "04_end_to_end_pipeline.ipynb",
    "Add-Punctuation-and-Translate.ipynb"
]

for notebook_name in notebooks:

    notebook_path = PROJECT_DIR / notebook_name

    print("\n" + "=" * 80)
    print("NOTEBOOK:", notebook_name)
    print("=" * 80)

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook["cells"]):

        source = "".join(cell["source"]).strip()

        if not source:
            continue

        if cell["cell_type"] == "markdown":
            print(f"\n[MARKDOWN CELL {i}]")
            print(source[:700])

        elif cell["cell_type"] == "code":
            # Print code cells, excluding excessively long output.
            print(f"\n[CODE CELL {i}]")
            print(source[:1000])

            if len(source) > 1000:
                print("... [remaining code omitted]")



NOTEBOOK: 02_sentence_ordering.ipynb

[CODE CELL 0]
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"

ORDERING_DATA_PATH = DATA_DIR / "ordering_processed"

[CODE CELL 1]
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

[CODE CELL 2]
print("Project exists:", PROJECT_DIR.exists())
print("Ordering dataset exists:", ORDERING_DATA_PATH.exists())
print("Model checkpoint exists:",
      (MODEL_DIR / "sentence_ordering_transformer" / "model_state_dict.pt").exists())

[CODE CELL 3]
from pathlib import Path
from datasets import load_from_disk

PROJECT_DIR = Path(r"E:\punctuation_project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

ORDERING_PATH = PROCESSED_DIR / "ordering_processed"

print("Dataset path:", O

In [37]:

data_dir = PROJECT_DIR / "data"

print("DATA DIRECTORY STRUCTURE")

for path in sorted(data_dir.rglob("*")):
    if path.is_file():
        relative = path.relative_to(PROJECT_DIR)
        print(relative)


DATA DIRECTORY STRUCTURE
data\processed\ordering_indicbert_processed\dataset_dict.json
data\processed\ordering_indicbert_processed\test\data-00000-of-00001.arrow
data\processed\ordering_indicbert_processed\test\dataset_info.json
data\processed\ordering_indicbert_processed\test\state.json
data\processed\ordering_indicbert_processed\train\data-00000-of-00001.arrow
data\processed\ordering_indicbert_processed\train\dataset_info.json
data\processed\ordering_indicbert_processed\train\state.json
data\processed\ordering_indicbert_processed\validation\data-00000-of-00001.arrow
data\processed\ordering_indicbert_processed\validation\dataset_info.json
data\processed\ordering_indicbert_processed\validation\state.json
data\processed\ordering_indicbert_sentence_processed\dataset_dict.json
data\processed\ordering_indicbert_sentence_processed\test\data-00000-of-00001.arrow
data\processed\ordering_indicbert_sentence_processed\test\dataset_info.json
data\processed\ordering_indicbert_sentence_processed\te